# Chapter 10 — Reduction

Runs every program of `ec2/reduction` on a Colab GPU. First choose **Runtime → Change runtime type → GPU (T4 is enough)**, then **Runtime → Run all**.

The theory is in `ec2/reduction/README.md`. The same `.cu` files also run on a laptop without a GPU: `clang++ -std=c++17 -O2 -pthread -x c++ -I../common FILE.cu -o FILE && ./FILE`.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

The compatibility header: plain CUDA under `nvcc`, a CPU emulation under `g++`/`clang++`.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

inline const char* cudaGetErrorString(cudaError_t) {
  return "no error";
}

inline cudaError_t cudaGetLastError() {
  return cudaSuccess;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_bike_totals.cu`

In [ ]:
%%writefile bikeshare_bike_totals.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <string>
#include <vector>

constexpr unsigned BLOCK_DIM = 256;
constexpr unsigned WARP_SIZE = 32;
constexpr unsigned WARPS_PER_BLOCK = BLOCK_DIM / WARP_SIZE;
constexpr unsigned FULL_WARP_MASK = 0xffffffffu;
constexpr unsigned NO_INDEX = 0xffffffffu;
constexpr unsigned TOP_EARNER_SLOT = 0;
constexpr unsigned MOST_USED_SLOT = 1;

enum class TripStatus : int { Completed = 0, StillRiding = 1, FalseStart = 2 };

#ifndef CPU_TRIPS
#define CPU_TRIPS (1u << 15)
#endif
#ifndef CPU_BIKES
#define CPU_BIKES 16u
#endif
const unsigned TRIP_COUNT = BY_BACKEND(1u << 24, CPU_TRIPS);
const unsigned BIKE_COUNT = BY_BACKEND(4096u, CPU_BIKES);
const int TIMED_REPETITIONS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

struct BikeSum {
  unsigned long long distanceM;
  unsigned long long revenueCents;
  unsigned trips;
};

__host__ __device__ inline BikeSum sumIdentity() {
  return BikeSum{0ull, 0ull, 0u};
}

__host__ __device__ inline BikeSum combine(const BikeSum& a, const BikeSum& b) {
  return BikeSum{a.distanceM + b.distanceM, a.revenueCents + b.revenueCents, a.trips + b.trips};
}

__device__ inline BikeSum shuffleDownSum(const BikeSum& v, unsigned delta) {
  BikeSum r;
  r.distanceM = __shfl_down_sync(FULL_WARP_MASK, v.distanceM, delta);
  r.revenueCents = __shfl_down_sync(FULL_WARP_MASK, v.revenueCents, delta);
  r.trips = __shfl_down_sync(FULL_WARP_MASK, v.trips, delta);
  return r;
}

__device__ inline BikeSum warpReduceSum(BikeSum v) {
  for (unsigned stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    v = combine(v, shuffleDownSum(v, stride));
  }
  return v;
}

__global__ void BikeTotalsKernel(const TripStatus* status,
                                 const int* distanceM,
                                 const int* fareCents,
                                 const unsigned* offset,
                                 unsigned long long* distance,
                                 unsigned long long* revenue,
                                 unsigned* trips) {
  const unsigned bike = blockIdx.x;
  const unsigned warp = threadIdx.x / WARP_SIZE;
  const unsigned lane = threadIdx.x % WARP_SIZE;

  BikeSum partial = sumIdentity();
  for (unsigned i = offset[bike] + threadIdx.x; i < offset[bike + 1]; i += BLOCK_DIM) {
    if (status[i] == TripStatus::Completed) {
      const BikeSum trip{static_cast<unsigned long long>(distanceM[i]),
                         static_cast<unsigned long long>(fareCents[i]),
                         1u};
      partial = combine(partial, trip);
    }
  }

  __shared__ BikeSum warp_s[WARPS_PER_BLOCK];
  partial = warpReduceSum(partial);
  if (lane == 0) {
    warp_s[warp] = partial;
  }
  __syncthreads();

  if (warp == 0) {
    const bool laneHasWarpResult = lane < WARPS_PER_BLOCK;
    BikeSum total = laneHasWarpResult ? warp_s[lane] : sumIdentity();
    total = warpReduceSum(total);
    if (threadIdx.x == 0) {
      distance[bike] = total.distanceM;
      revenue[bike] = total.revenueCents;
      trips[bike] = total.trips;
    }
  }
}

struct Best {
  unsigned long long value;
  unsigned index;
};

__host__ __device__ inline Best bestIdentity() {
  return Best{0ull, NO_INDEX};
}

__host__ __device__ inline bool beats(const Best& a, const Best& b) {
  return a.value > b.value || (a.value == b.value && a.index < b.index);
}

__host__ __device__ inline Best better(const Best& a, const Best& b) {
  return beats(a, b) ? a : b;
}

__device__ inline Best shuffleDownBest(const Best& v, unsigned delta) {
  Best r;
  r.value = __shfl_down_sync(FULL_WARP_MASK, v.value, delta);
  r.index = __shfl_down_sync(FULL_WARP_MASK, v.index, delta);
  return r;
}

__device__ inline Best warpReduceBest(Best v) {
  for (unsigned stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    v = better(v, shuffleDownBest(v, stride));
  }
  return v;
}

__global__ void ArgMaxKernel(const unsigned long long* revenue,
                             const unsigned long long* distance,
                             unsigned bikes,
                             Best* out) {
  const unsigned warp = threadIdx.x / WARP_SIZE;
  const unsigned lane = threadIdx.x % WARP_SIZE;

  Best topEarner = bestIdentity();
  Best mostUsed = bestIdentity();
  for (unsigned b = threadIdx.x; b < bikes; b += BLOCK_DIM) {
    topEarner = better(topEarner, Best{revenue[b], b});
    mostUsed = better(mostUsed, Best{distance[b], b});
  }

  __shared__ Best topEarner_s[WARPS_PER_BLOCK];
  __shared__ Best mostUsed_s[WARPS_PER_BLOCK];
  topEarner = warpReduceBest(topEarner);
  mostUsed = warpReduceBest(mostUsed);
  if (lane == 0) {
    topEarner_s[warp] = topEarner;
    mostUsed_s[warp] = mostUsed;
  }
  __syncthreads();

  if (warp == 0) {
    const bool laneHasWarpResult = lane < WARPS_PER_BLOCK;
    Best blockEarner = laneHasWarpResult ? topEarner_s[lane] : bestIdentity();
    Best blockMostUsed = laneHasWarpResult ? mostUsed_s[lane] : bestIdentity();
    blockEarner = warpReduceBest(blockEarner);
    blockMostUsed = warpReduceBest(blockMostUsed);
    if (threadIdx.x == 0) {
      out[TOP_EARNER_SLOT] = blockEarner;
      out[MOST_USED_SLOT] = blockMostUsed;
    }
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_GAMMA;
    uint64_t mixed = state_;
    mixed = (mixed ^ (mixed >> 30)) * FIRST_MULTIPLIER;
    mixed = (mixed ^ (mixed >> 27)) * SECOND_MULTIPLIER;
    return mixed ^ (mixed >> 31);
  }

  int range(int lowest, int highest) {
    const uint64_t choices = static_cast<uint64_t>(highest - lowest + 1);
    return lowest + static_cast<int>(next() % choices);
  }

 private:
  static constexpr uint64_t GOLDEN_GAMMA = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

constexpr uint64_t DATA_SEED = 2025;
constexpr int PER_MILLE = 1000;
constexpr int STILL_RIDING_PER_MILLE = 30;
constexpr int FALSE_START_PER_MILLE = 50;
constexpr int DOWNTOWN_CHANCES_IN_TEN = 3;
constexpr int SHORTEST_TRIP_M = 300;
constexpr int LONGEST_TRIP_M = 8000;
constexpr int MINUTE_MS = 60000;
constexpr int MS_PER_METRE_AT_15_KMH = 240;
constexpr int FARE_STEP_MS = 10 * MINUTE_MS;
constexpr int BASE_FARE_CENTS = 300;
constexpr int CENTS_PER_FARE_STEP = 100;
constexpr int METRES_PER_KM = 1000;
constexpr unsigned CENTS_PER_BOLIVIANO = 100;
constexpr unsigned TOP_EARNERS_SHOWN = 5;

constexpr uint64_t TIE_TEST_SEED = 99;
constexpr unsigned TIE_TEST_BIKES = 3000;
constexpr unsigned long long RANDOM_TOTAL_LIMIT = 1000000;
constexpr unsigned long long TIED_MAXIMUM = 2000000;
constexpr unsigned TIED_EARNERS[] = {2999, 513, 259};
constexpr unsigned TIED_MOST_USED[] = {2050, 1795, 1000};

struct Trips {
  std::vector<TripStatus> status;
  std::vector<unsigned> bike;
  std::vector<int> distanceM;
  std::vector<int> fareCents;
};

struct TripsByBike {
  std::vector<unsigned> offset;
  std::vector<TripStatus> status;
  std::vector<int> distanceM;
  std::vector<int> fareCents;
};

struct BikeTotals {
  std::vector<unsigned long long> distanceM;
  std::vector<unsigned long long> revenueCents;
  std::vector<unsigned> trips;
};

struct ArgMaxResult {
  Best topEarner;
  Best mostUsed;
  double ms;
};

struct CpuReference {
  BikeTotals totals;
  Best topEarner;
  Best mostUsed;
  double ms;
};

static bool operator==(const BikeTotals& a, const BikeTotals& b) {
  return a.distanceM == b.distanceM && a.revenueCents == b.revenueCents && a.trips == b.trips;
}

static bool operator==(const Best& a, const Best& b) {
  return a.value == b.value && a.index == b.index;
}

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

static int fareCentsForRide(int rideMs) {
  const int startedFareSteps = (rideMs + FARE_STEP_MS - 1) / FARE_STEP_MS;
  return BASE_FARE_CENTS + startedFareSteps * CENTS_PER_FARE_STEP;
}

static int syntheticRideMs(SplitMix64& random, int distanceM) {
  const int pedallingMs = distanceM * MS_PER_METRE_AT_15_KMH;
  const int trafficLightsMs = random.range(1 * MINUTE_MS, 12 * MINUTE_MS);
  return pedallingMs + trafficLightsMs;
}

static TripStatus syntheticStatus(SplitMix64& random) {
  const int outcome = random.range(0, PER_MILLE - 1);
  if (outcome < STILL_RIDING_PER_MILLE) {
    return TripStatus::StillRiding;
  }
  if (outcome < STILL_RIDING_PER_MILLE + FALSE_START_PER_MILLE) {
    return TripStatus::FalseStart;
  }
  return TripStatus::Completed;
}

static unsigned syntheticBike(SplitMix64& random, unsigned bikes) {
  unsigned bike = static_cast<unsigned>(random.range(0, static_cast<int>(bikes) - 1));
  const bool docksDowntown = random.range(0, 9) < DOWNTOWN_CHANCES_IN_TEN;
  if (docksDowntown) {
    bike /= 2;
  }
  return bike;
}

static Trips syntheticDay(unsigned count, unsigned bikes, uint64_t seed) {
  SplitMix64 random(seed);
  Trips trips;
  for (unsigned i = 0; i < count; ++i) {
    const TripStatus status = syntheticStatus(random);
    const unsigned bike = syntheticBike(random, bikes);
    const bool isCompleted = status == TripStatus::Completed;
    const int distanceM = isCompleted ? random.range(SHORTEST_TRIP_M, LONGEST_TRIP_M) : 0;
    const int fareCents = isCompleted ? fareCentsForRide(syntheticRideMs(random, distanceM)) : 0;

    trips.status.push_back(status);
    trips.bike.push_back(bike);
    trips.distanceM.push_back(distanceM);
    trips.fareCents.push_back(fareCents);
  }
  return trips;
}

static std::vector<unsigned> bikeOffsets(const Trips& trips, unsigned bikes) {
  std::vector<unsigned> offset(bikes + 1, 0);
  for (unsigned bike : trips.bike) {
    ++offset[bike + 1];
  }
  for (unsigned b = 0; b < bikes; ++b) {
    offset[b + 1] += offset[b];
  }
  return offset;
}

static TripsByBike groupByBike(const Trips& trips, unsigned bikes) {
  TripsByBike grouped;
  grouped.offset = bikeOffsets(trips, bikes);
  const unsigned groupedCount = grouped.offset[bikes];
  grouped.status.resize(groupedCount);
  grouped.distanceM.resize(groupedCount);
  grouped.fareCents.resize(groupedCount);

  std::vector<unsigned> nextSlot(grouped.offset.begin(), grouped.offset.end() - 1);
  for (size_t i = 0; i < trips.status.size(); ++i) {
    const unsigned slot = nextSlot[trips.bike[i]]++;
    grouped.status[slot] = trips.status[i];
    grouped.distanceM[slot] = trips.distanceM[i];
    grouped.fareCents[slot] = trips.fareCents[i];
  }
  return grouped;
}

static void printSliceLengths(const TripsByBike& grouped, unsigned bikes) {
  unsigned shortest = grouped.offset[bikes];
  unsigned longest = 0;
  for (unsigned b = 0; b < bikes; ++b) {
    const unsigned length = grouped.offset[b + 1] - grouped.offset[b];
    shortest = std::min(shortest, length);
    longest = std::max(longest, length);
  }
  std::printf("slices per bike: %u to %u trips (each block walks its slice %u at a time)\n",
              shortest,
              longest,
              BLOCK_DIM);
}

static Best cpuArgMax(const std::vector<unsigned long long>& values) {
  Best best = bestIdentity();
  for (unsigned b = 0; b < values.size(); ++b) {
    best = better(best, Best{values[b], b});
  }
  return best;
}

static CpuReference computeCpuReference(const Trips& trips, unsigned bikes) {
  Timer timer;
  CpuReference reference;
  reference.totals.distanceM.assign(bikes, 0);
  reference.totals.revenueCents.assign(bikes, 0);
  reference.totals.trips.assign(bikes, 0);
  for (size_t i = 0; i < trips.status.size(); ++i) {
    if (trips.status[i] != TripStatus::Completed) {
      continue;
    }
    const unsigned bike = trips.bike[i];
    reference.totals.distanceM[bike] += static_cast<unsigned long long>(trips.distanceM[i]);
    reference.totals.revenueCents[bike] += static_cast<unsigned long long>(trips.fareCents[i]);
    ++reference.totals.trips[bike];
  }
  reference.topEarner = cpuArgMax(reference.totals.revenueCents);
  reference.mostUsed = cpuArgMax(reference.totals.distanceM);
  reference.ms = timer.ms();
  return reference;
}

template <class T>
static T* allocateOnDevice(size_t count) {
  const size_t nonEmptyCount = std::max<size_t>(count, 1);
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, nonEmptyCount * sizeof(T)));
  return pointer;
}

template <class T>
static T* deviceCopyOf(const std::vector<T>& values) {
  T* pointer = allocateOnDevice<T>(values.size());
  if (!values.empty()) {
    CHECK(cudaMemcpy(pointer, values.data(), values.size() * sizeof(T), cudaMemcpyHostToDevice));
  }
  return pointer;
}

template <class T>
static std::vector<T> hostCopyOf(const T* values, size_t count) {
  std::vector<T> copy(count);
  CHECK(cudaMemcpy(copy.data(), values, count * sizeof(T), cudaMemcpyDeviceToHost));
  return copy;
}

template <class Work>
static double millisecondsPerRun(Work runOnce) {
  if (WARM_UP_BEFORE_TIMING) {
    runOnce();
  }
  Timer timer;
  for (int repetition = 0; repetition < TIMED_REPETITIONS; ++repetition) {
    runOnce();
  }
  const double milliseconds = timer.ms() / TIMED_REPETITIONS;
  CHECK(cudaGetLastError());
  return milliseconds;
}

static BikeTotals runBikeTotalsKernel(const TripsByBike& grouped, unsigned bikes, double& ms) {
  TripStatus* status = deviceCopyOf(grouped.status);
  int* distanceM = deviceCopyOf(grouped.distanceM);
  int* fareCents = deviceCopyOf(grouped.fareCents);
  unsigned* offset = deviceCopyOf(grouped.offset);
  unsigned long long* distance = allocateOnDevice<unsigned long long>(bikes);
  unsigned long long* revenue = allocateOnDevice<unsigned long long>(bikes);
  unsigned* trips = allocateOnDevice<unsigned>(bikes);

  ms = millisecondsPerRun([&] {
    launch(BikeTotalsKernel,
           dim3(bikes),
           dim3(BLOCK_DIM),
           status,
           distanceM,
           fareCents,
           offset,
           distance,
           revenue,
           trips);
  });

  BikeTotals totals;
  totals.distanceM = hostCopyOf(distance, bikes);
  totals.revenueCents = hostCopyOf(revenue, bikes);
  totals.trips = hostCopyOf(trips, bikes);

  CHECK(cudaFree(status));
  CHECK(cudaFree(distanceM));
  CHECK(cudaFree(fareCents));
  CHECK(cudaFree(offset));
  CHECK(cudaFree(distance));
  CHECK(cudaFree(revenue));
  CHECK(cudaFree(trips));
  return totals;
}

static ArgMaxResult runArgMaxKernel(const std::vector<unsigned long long>& revenue,
                                    const std::vector<unsigned long long>& distance) {
  const unsigned bikes = static_cast<unsigned>(revenue.size());
  unsigned long long* deviceRevenue = deviceCopyOf(revenue);
  unsigned long long* deviceDistance = deviceCopyOf(distance);
  Best* out = allocateOnDevice<Best>(2);

  ArgMaxResult result;
  result.ms = millisecondsPerRun([&] {
    launch(ArgMaxKernel, dim3(1), dim3(BLOCK_DIM), deviceRevenue, deviceDistance, bikes, out);
  });
  const std::vector<Best> winners = hostCopyOf(out, 2);
  result.topEarner = winners[TOP_EARNER_SLOT];
  result.mostUsed = winners[MOST_USED_SLOT];

  CHECK(cudaFree(deviceRevenue));
  CHECK(cudaFree(deviceDistance));
  CHECK(cudaFree(out));
  return result;
}

static void printTimingTable(
    double cpuMs, unsigned bikes, double totalsMs, bool totalsOk, double argMaxMs, bool argMaxOk) {
  std::printf("\n%-36s %7s %10s  %s\n", "kernel", "blocks", "ms", "check");
  std::printf(
      "%-36s %7s %10.3f  %s\n", "CPU sequential (totals + argmax)", "-", cpuMs, "reference");
  std::printf("%-36s %7u %10.3f  %s\n",
              "BikeTotalsKernel (segmented)",
              bikes,
              totalsMs,
              passOrFail(totalsOk));
  std::printf("%-36s %7u %10.3f  %s\n",
              "ArgMaxKernel (value, index) pairs",
              1u,
              argMaxMs,
              passOrFail(argMaxOk));
}

static std::vector<unsigned> bikesByRevenue(const std::vector<unsigned long long>& revenue) {
  std::vector<unsigned> ranking(revenue.size());
  for (unsigned b = 0; b < ranking.size(); ++b) {
    ranking[b] = b;
  }
  std::sort(ranking.begin(), ranking.end(), [&](unsigned a, unsigned b) {
    return beats(Best{revenue[a], a}, Best{revenue[b], b});
  });
  return ranking;
}

static void printTopEarners(const BikeTotals& totals, const ArgMaxResult& winners) {
  const std::vector<unsigned> ranking = bikesByRevenue(totals.revenueCents);
  const size_t shown = std::min<size_t>(TOP_EARNERS_SHOWN, ranking.size());
  std::printf("\n%-8s %8s %14s %12s\n", "bike", "trips", "revenue (Bs)", "ridden (km)");
  for (size_t k = 0; k < shown; ++k) {
    const unsigned b = ranking[k];
    const std::string name = "b" + std::to_string(b);
    std::printf("%-8s %8u %10llu.%02llu %12.1f\n",
                name.c_str(),
                totals.trips[b],
                totals.revenueCents[b] / CENTS_PER_BOLIVIANO,
                totals.revenueCents[b] % CENTS_PER_BOLIVIANO,
                static_cast<double>(totals.distanceM[b]) / METRES_PER_KM);
  }
  std::printf("top earner: b%u, Bs %llu.%02llu\n",
              winners.topEarner.index,
              winners.topEarner.value / CENTS_PER_BOLIVIANO,
              winners.topEarner.value % CENTS_PER_BOLIVIANO);
  std::printf("most used:  b%u, %.1f km\n",
              winners.mostUsed.index,
              static_cast<double>(winners.mostUsed.value) / METRES_PER_KM);
}

template <size_t COUNT>
static unsigned lowestOf(const unsigned (&indices)[COUNT]) {
  return *std::min_element(indices, indices + COUNT);
}

static bool checkInjectedTies() {
  std::vector<unsigned long long> revenue(TIE_TEST_BIKES);
  std::vector<unsigned long long> distance(TIE_TEST_BIKES);
  SplitMix64 random(TIE_TEST_SEED);
  for (unsigned b = 0; b < TIE_TEST_BIKES; ++b) {
    revenue[b] = random.next() % RANDOM_TOTAL_LIMIT;
    distance[b] = random.next() % RANDOM_TOTAL_LIMIT;
  }
  for (unsigned bike : TIED_EARNERS) {
    revenue[bike] = TIED_MAXIMUM;
  }
  for (unsigned bike : TIED_MOST_USED) {
    distance[bike] = TIED_MAXIMUM;
  }

  const ArgMaxResult tie = runArgMaxKernel(revenue, distance);
  const bool ok = tie.topEarner == cpuArgMax(revenue) && tie.mostUsed == cpuArgMax(distance) &&
                  tie.topEarner.index == lowestOf(TIED_EARNERS) &&
                  tie.mostUsed.index == lowestOf(TIED_MOST_USED);
  std::printf(
      "\ntie test: revenue tied at b%u, b%u, b%u -> b%u; distance tied at b%u, b%u, b%u -> b%u  "
      "%s\n",
      TIED_EARNERS[0],
      TIED_EARNERS[1],
      TIED_EARNERS[2],
      tie.topEarner.index,
      TIED_MOST_USED[0],
      TIED_MOST_USED[1],
      TIED_MOST_USED[2],
      tie.mostUsed.index,
      passOrFail(ok));
  return ok;
}

static bool checkAllZeroTotals(unsigned bikes) {
  const std::vector<unsigned long long> zeros(bikes, 0);
  const ArgMaxResult zero = runArgMaxKernel(zeros, zeros);
  const bool ok =
      zero.topEarner.index == 0 && zero.mostUsed.index == 0 && zero.topEarner.value == 0;
  std::printf("tie test: all bikes at 0 -> b%u and b%u  %s\n",
              zero.topEarner.index,
              zero.mostUsed.index,
              passOrFail(ok));
  return ok;
}

int main() {
  std::printf(
      "Per-bike totals, the most used bike and the top earner (chapter 10 use case), "
      "backend: %s\n",
      CUDA_BACKEND);
  const unsigned bikes = BIKE_COUNT;
  const Trips trips = syntheticDay(TRIP_COUNT, bikes, DATA_SEED);
  std::printf("source: synthetic, %u trips, %u bikes (seed 2025), %u threads per block\n",
              TRIP_COUNT,
              bikes,
              BLOCK_DIM);

  const TripsByBike grouped = groupByBike(trips, bikes);
  printSliceLengths(grouped, bikes);
  const CpuReference reference = computeCpuReference(trips, bikes);

  double totalsMs = 0;
  const BikeTotals totals = runBikeTotalsKernel(grouped, bikes, totalsMs);
  const bool totalsOk = totals == reference.totals;
  const ArgMaxResult winners = runArgMaxKernel(totals.revenueCents, totals.distanceM);
  const bool argMaxOk =
      winners.topEarner == reference.topEarner && winners.mostUsed == reference.mostUsed;

  printTimingTable(reference.ms, bikes, totalsMs, totalsOk, winners.ms, argMaxOk);
  printTopEarners(totals, winners);
  const bool injectedTiesOk = checkInjectedTies();
  const bool allZeroOk = checkAllZeroTotals(bikes);

  const bool ok = totalsOk && argMaxOk && injectedTiesOk && allZeroOk;
  std::printf("\n%s\n", ok ? "ALL PASS" : "SOME FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_bike_totals.cu -o bikeshare_bike_totals && ./bikeshare_bike_totals

## `bikeshare_daily_summary.cu`

In [ ]:
%%writefile bikeshare_daily_summary.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

constexpr unsigned BLOCK_DIM = 256;
constexpr unsigned COARSE_FACTOR = 16;
constexpr unsigned TRIPS_PER_SUMMARY_BLOCK = 2 * COARSE_FACTOR * BLOCK_DIM;
constexpr int MAX_IDENTITY_FOR_DURATIONS = 0;

constexpr int MINUTE_MS = 60000;
constexpr int SECOND_MS = 1000;
constexpr int SECONDS_PER_MINUTE = 60;
constexpr int INCLUDED_RIDE_MS = 45 * MINUTE_MS;
constexpr int FARE_STEP_MS = 10 * MINUTE_MS;
constexpr int BASE_FARE_CENTS = 300;
constexpr int CENTS_PER_FARE_STEP = 100;
constexpr unsigned long long CENTS_PER_BOLIVIANO = 100;

enum class TripStatus : int { Returned = 0, StillRiding = 1, SentToRepair = 2 };

#ifndef CPU_TRIPS
#define CPU_TRIPS (1u << 18)
#endif
#ifndef CPU_BIKES
#define CPU_BIKES 240u
#endif
const unsigned SYNTHETIC_TRIPS = BY_BACKEND(1u << 24, CPU_TRIPS);
const unsigned SYNTHETIC_BIKES = BY_BACKEND(480u, CPU_BIKES);
constexpr unsigned STATION_COUNT = 48;
constexpr uint64_t SYNTHETIC_SEED = 2025;
const int TIMED_REPETITIONS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

constexpr int PER_MILLE = 1000;
constexpr int STILL_RIDING_PER_MILLE = 30;
constexpr int SENT_TO_REPAIR_PER_MILLE = 15;
constexpr int LEISURE_RIDE_PERCENT = 6;
constexpr int EBIKE_EVERY = 4;
constexpr int EBIKE_EXTRA_RIDE_PERCENT = 25;

struct Totals {
  unsigned returned;
  unsigned stillRiding;
  unsigned sentToRepair;
  unsigned late;
  unsigned long long fareCents;
  unsigned long long rideMs;
  int maxRideMs;
};

struct BikeTotals {
  unsigned rides;
  unsigned repairs;
  unsigned long long rideMs;
  int maxRideMs;
};

__host__ __device__ inline Totals emptyTotals() {
  return Totals{0u, 0u, 0u, 0u, 0ull, 0ull, MAX_IDENTITY_FOR_DURATIONS};
}

__host__ __device__ inline BikeTotals emptyBikeTotals() {
  return BikeTotals{0u, 0u, 0ull, MAX_IDENTITY_FOR_DURATIONS};
}

__host__ __device__ inline bool isLate(int rideMs) {
  return rideMs > INCLUDED_RIDE_MS;
}

__host__ __device__ inline void addTrip(Totals& totals,
                                        TripStatus status,
                                        int rideMs,
                                        int fareCents) {
  if (status == TripStatus::StillRiding) {
    ++totals.stillRiding;
    return;
  }

  if (status == TripStatus::Returned) {
    ++totals.returned;
  } else {
    ++totals.sentToRepair;
  }
  if (isLate(rideMs)) {
    ++totals.late;
  }
  totals.fareCents += static_cast<unsigned long long>(fareCents);
  totals.rideMs += static_cast<unsigned long long>(rideMs);
  totals.maxRideMs = imax(totals.maxRideMs, rideMs);
}

__host__ __device__ inline void addBikeTrip(BikeTotals& totals, TripStatus status, int rideMs) {
  if (status == TripStatus::StillRiding) {
    return;
  }

  ++totals.rides;
  if (status == TripStatus::SentToRepair) {
    ++totals.repairs;
  }
  totals.rideMs += static_cast<unsigned long long>(rideMs);
  totals.maxRideMs = imax(totals.maxRideMs, rideMs);
}

__global__ void SummaryKernel(
    const TripStatus* status, const int* rideMs, const int* fareCents, unsigned n, Totals* totals) {
  __shared__ unsigned returned_s[BLOCK_DIM];
  __shared__ unsigned stillRiding_s[BLOCK_DIM];
  __shared__ unsigned sentToRepair_s[BLOCK_DIM];
  __shared__ unsigned late_s[BLOCK_DIM];
  __shared__ unsigned long long fareCents_s[BLOCK_DIM];
  __shared__ unsigned long long rideMs_s[BLOCK_DIM];
  __shared__ int maxRideMs_s[BLOCK_DIM];

  unsigned t = threadIdx.x;
  unsigned segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  Totals partial = emptyTotals();
  for (unsigned tile = 0; tile < 2 * COARSE_FACTOR; ++tile) {
    unsigned i = segment + t + tile * BLOCK_DIM;
    if (i < n) {
      addTrip(partial, status[i], rideMs[i], fareCents[i]);
    }
  }
  returned_s[t] = partial.returned;
  stillRiding_s[t] = partial.stillRiding;
  sentToRepair_s[t] = partial.sentToRepair;
  late_s[t] = partial.late;
  fareCents_s[t] = partial.fareCents;
  rideMs_s[t] = partial.rideMs;
  maxRideMs_s[t] = partial.maxRideMs;

  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      returned_s[t] += returned_s[t + stride];
      stillRiding_s[t] += stillRiding_s[t + stride];
      sentToRepair_s[t] += sentToRepair_s[t + stride];
      late_s[t] += late_s[t + stride];
      fareCents_s[t] += fareCents_s[t + stride];
      rideMs_s[t] += rideMs_s[t + stride];
      maxRideMs_s[t] = imax(maxRideMs_s[t], maxRideMs_s[t + stride]);
    }
  }

  if (t == 0) {
    atomicAdd(&totals->returned, returned_s[0]);
    atomicAdd(&totals->stillRiding, stillRiding_s[0]);
    atomicAdd(&totals->sentToRepair, sentToRepair_s[0]);
    atomicAdd(&totals->late, late_s[0]);
    atomicAdd(&totals->fareCents, fareCents_s[0]);
    atomicAdd(&totals->rideMs, rideMs_s[0]);
    atomicMax(&totals->maxRideMs, maxRideMs_s[0]);
  }
}

__global__ void PerBikeKernel(const TripStatus* status,
                              const int* rideMs,
                              const unsigned* offset,
                              unsigned* rides,
                              unsigned* repairs,
                              unsigned long long* totalRideMs,
                              int* maxRideMs) {
  __shared__ unsigned rides_s[BLOCK_DIM];
  __shared__ unsigned repairs_s[BLOCK_DIM];
  __shared__ unsigned long long rideMs_s[BLOCK_DIM];
  __shared__ int maxRideMs_s[BLOCK_DIM];

  unsigned t = threadIdx.x;
  unsigned bike = blockIdx.x;
  BikeTotals partial = emptyBikeTotals();
  for (unsigned i = offset[bike] + t; i < offset[bike + 1]; i += blockDim.x) {
    addBikeTrip(partial, status[i], rideMs[i]);
  }
  rides_s[t] = partial.rides;
  repairs_s[t] = partial.repairs;
  rideMs_s[t] = partial.rideMs;
  maxRideMs_s[t] = partial.maxRideMs;

  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      rides_s[t] += rides_s[t + stride];
      repairs_s[t] += repairs_s[t + stride];
      rideMs_s[t] += rideMs_s[t + stride];
      maxRideMs_s[t] = imax(maxRideMs_s[t], maxRideMs_s[t + stride]);
    }
  }

  if (t == 0) {
    rides[bike] = rides_s[0];
    repairs[bike] = repairs_s[0];
    totalRideMs[bike] = rideMs_s[0];
    maxRideMs[bike] = maxRideMs_s[0];
  }
}

struct Trips {
  std::vector<TripStatus> status;
  std::vector<unsigned> bike;
  std::vector<int> rideMs;
  std::vector<int> fareCents;
};

struct TripsByBike {
  std::vector<unsigned> offset;
  std::vector<TripStatus> status;
  std::vector<int> rideMs;
};

struct SummaryRun {
  Totals totals;
  unsigned blocks;
  double ms;
};

struct PerBikeRun {
  std::vector<BikeTotals> bikes;
  double ms;
};

struct CpuReference {
  Totals totals;
  std::vector<BikeTotals> bikes;
  double ms;
};

static bool operator==(const Totals& a, const Totals& b) {
  return a.returned == b.returned && a.stillRiding == b.stillRiding &&
         a.sentToRepair == b.sentToRepair && a.late == b.late && a.fareCents == b.fareCents &&
         a.rideMs == b.rideMs && a.maxRideMs == b.maxRideMs;
}

static bool operator==(const BikeTotals& a, const BikeTotals& b) {
  return a.rides == b.rides && a.repairs == b.repairs && a.rideMs == b.rideMs &&
         a.maxRideMs == b.maxRideMs;
}

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_GAMMA;
    uint64_t mixed = state_;
    mixed = (mixed ^ (mixed >> 30)) * FIRST_MULTIPLIER;
    mixed = (mixed ^ (mixed >> 27)) * SECOND_MULTIPLIER;
    return mixed ^ (mixed >> 31);
  }

  int range(int lowest, int highest) {
    const uint64_t choices = static_cast<uint64_t>(highest - lowest + 1);
    return lowest + static_cast<int>(next() % choices);
  }

 private:
  static constexpr uint64_t GOLDEN_GAMMA = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

static int fareCentsForRide(int rideMs) {
  const int startedFareSteps = (rideMs + FARE_STEP_MS - 1) / FARE_STEP_MS;
  return BASE_FARE_CENTS + startedFareSteps * CENTS_PER_FARE_STEP;
}

static bool isEBike(unsigned bike) {
  return bike % EBIKE_EVERY == 0;
}

static int syntheticRideMs(SplitMix64& random, unsigned bike) {
  int rideMs = random.range(4 * MINUTE_MS, 25 * MINUTE_MS);
  const bool isLeisureRide = random.range(0, 99) < LEISURE_RIDE_PERCENT;
  if (isLeisureRide) {
    rideMs = random.range(30 * MINUTE_MS, 110 * MINUTE_MS);
  }
  if (isEBike(bike)) {
    rideMs += rideMs / 100 * EBIKE_EXTRA_RIDE_PERCENT;
  }
  return rideMs;
}

static TripStatus syntheticStatus(SplitMix64& random) {
  const int outcome = random.range(0, PER_MILLE - 1);
  if (outcome < STILL_RIDING_PER_MILLE) {
    return TripStatus::StillRiding;
  }
  if (outcome < STILL_RIDING_PER_MILLE + SENT_TO_REPAIR_PER_MILLE) {
    return TripStatus::SentToRepair;
  }
  return TripStatus::Returned;
}

static Trips syntheticTrips(unsigned count, unsigned bikes, uint64_t seed) {
  SplitMix64 random(seed);
  Trips trips;
  for (unsigned i = 0; i < count; ++i) {
    const TripStatus status = syntheticStatus(random);
    const unsigned bike = static_cast<unsigned>(random.range(0, static_cast<int>(bikes) - 1));
    const bool hasEnded = status != TripStatus::StillRiding;
    const int rideMs = hasEnded ? syntheticRideMs(random, bike) : 0;
    const int fareCents = hasEnded ? fareCentsForRide(rideMs) : 0;

    trips.status.push_back(status);
    trips.bike.push_back(bike);
    trips.rideMs.push_back(rideMs);
    trips.fareCents.push_back(fareCents);
  }
  return trips;
}

static std::vector<unsigned> bikeOffsets(const Trips& trips, unsigned bikes) {
  std::vector<unsigned> offset(bikes + 1, 0);
  for (unsigned bike : trips.bike) {
    ++offset[bike + 1];
  }
  for (unsigned b = 0; b < bikes; ++b) {
    offset[b + 1] += offset[b];
  }
  return offset;
}

static TripsByBike groupByBike(const Trips& trips, unsigned bikes) {
  TripsByBike grouped;
  grouped.offset = bikeOffsets(trips, bikes);
  const unsigned groupedCount = grouped.offset[bikes];
  grouped.status.resize(groupedCount);
  grouped.rideMs.resize(groupedCount);

  std::vector<unsigned> nextSlot(grouped.offset.begin(), grouped.offset.end() - 1);
  for (size_t i = 0; i < trips.status.size(); ++i) {
    const unsigned slot = nextSlot[trips.bike[i]]++;
    grouped.status[slot] = trips.status[i];
    grouped.rideMs[slot] = trips.rideMs[i];
  }
  return grouped;
}

static CpuReference computeCpuReference(const Trips& trips, unsigned bikes) {
  Timer timer;
  CpuReference reference;
  reference.totals = emptyTotals();
  reference.bikes.assign(bikes, emptyBikeTotals());
  for (size_t i = 0; i < trips.status.size(); ++i) {
    addTrip(reference.totals, trips.status[i], trips.rideMs[i], trips.fareCents[i]);
    addBikeTrip(reference.bikes[trips.bike[i]], trips.status[i], trips.rideMs[i]);
  }
  reference.ms = timer.ms();
  return reference;
}

template <class T>
static T* allocateOnDevice(size_t count) {
  const size_t nonEmptyCount = std::max<size_t>(count, 1);
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, nonEmptyCount * sizeof(T)));
  return pointer;
}

template <class T>
static T* deviceCopyOf(const std::vector<T>& values) {
  T* pointer = allocateOnDevice<T>(values.size());
  if (!values.empty()) {
    CHECK(cudaMemcpy(pointer, values.data(), values.size() * sizeof(T), cudaMemcpyHostToDevice));
  }
  return pointer;
}

template <class T>
static std::vector<T> hostCopyOf(const T* values, size_t count) {
  std::vector<T> copy(count);
  if (count > 0) {
    CHECK(cudaMemcpy(copy.data(), values, count * sizeof(T), cudaMemcpyDeviceToHost));
  }
  return copy;
}

template <class Work>
static double millisecondsPerRun(Work runOnce) {
  if (WARM_UP_BEFORE_TIMING) {
    runOnce();
  }
  Timer timer;
  for (int repetition = 0; repetition < TIMED_REPETITIONS; ++repetition) {
    runOnce();
  }
  const double milliseconds = timer.ms() / TIMED_REPETITIONS;
  CHECK(cudaGetLastError());
  return milliseconds;
}

static SummaryRun runSummaryKernel(const Trips& trips) {
  const unsigned n = static_cast<unsigned>(trips.status.size());
  TripStatus* status = deviceCopyOf(trips.status);
  int* rideMs = deviceCopyOf(trips.rideMs);
  int* fareCents = deviceCopyOf(trips.fareCents);
  Totals* totals = allocateOnDevice<Totals>(1);
  const Totals zero = emptyTotals();

  SummaryRun run;
  run.blocks = n > 0 ? cdiv(n, TRIPS_PER_SUMMARY_BLOCK) : 1;
  run.ms = millisecondsPerRun([&] {
    CHECK(cudaMemcpy(totals, &zero, sizeof(Totals), cudaMemcpyHostToDevice));
    launch(SummaryKernel, dim3(run.blocks), dim3(BLOCK_DIM), status, rideMs, fareCents, n, totals);
  });
  CHECK(cudaMemcpy(&run.totals, totals, sizeof(Totals), cudaMemcpyDeviceToHost));

  CHECK(cudaFree(status));
  CHECK(cudaFree(rideMs));
  CHECK(cudaFree(fareCents));
  CHECK(cudaFree(totals));
  return run;
}

static PerBikeRun runPerBikeKernel(const TripsByBike& grouped, unsigned bikes) {
  TripStatus* status = deviceCopyOf(grouped.status);
  int* rideMs = deviceCopyOf(grouped.rideMs);
  unsigned* offset = deviceCopyOf(grouped.offset);
  unsigned* rides = allocateOnDevice<unsigned>(bikes);
  unsigned* repairs = allocateOnDevice<unsigned>(bikes);
  unsigned long long* totalRideMs = allocateOnDevice<unsigned long long>(bikes);
  int* maxRideMs = allocateOnDevice<int>(bikes);

  PerBikeRun run;
  run.ms = millisecondsPerRun([&] {
    if (bikes > 0) {
      launch(PerBikeKernel,
             dim3(bikes),
             dim3(BLOCK_DIM),
             status,
             rideMs,
             offset,
             rides,
             repairs,
             totalRideMs,
             maxRideMs);
    }
  });

  const std::vector<unsigned> ridesCopy = hostCopyOf(rides, bikes);
  const std::vector<unsigned> repairsCopy = hostCopyOf(repairs, bikes);
  const std::vector<unsigned long long> rideMsCopy = hostCopyOf(totalRideMs, bikes);
  const std::vector<int> maxRideMsCopy = hostCopyOf(maxRideMs, bikes);
  for (unsigned b = 0; b < bikes; ++b) {
    run.bikes.push_back(BikeTotals{ridesCopy[b], repairsCopy[b], rideMsCopy[b], maxRideMsCopy[b]});
  }

  CHECK(cudaFree(status));
  CHECK(cudaFree(rideMs));
  CHECK(cudaFree(offset));
  CHECK(cudaFree(rides));
  CHECK(cudaFree(repairs));
  CHECK(cudaFree(totalRideMs));
  CHECK(cudaFree(maxRideMs));
  return run;
}

static unsigned long long ridesOfAllBikes(const std::vector<BikeTotals>& bikes) {
  unsigned long long total = 0;
  for (const BikeTotals& bike : bikes) {
    total += bike.rides;
  }
  return total;
}

static unsigned long long repairsOfAllBikes(const std::vector<BikeTotals>& bikes) {
  unsigned long long total = 0;
  for (const BikeTotals& bike : bikes) {
    total += bike.repairs;
  }
  return total;
}

static unsigned finishedTrips(const Totals& totals) {
  return totals.returned + totals.sentToRepair;
}

static const char* yesOrNo(bool holds) {
  return holds ? "yes" : "NO";
}

static void printTimingTable(const CpuReference& reference,
                             const SummaryRun& summary,
                             bool summaryOk,
                             const PerBikeRun& perBike,
                             bool perBikeOk) {
  std::printf("\n%-34s %7s %10s  %s\n", "kernel", "blocks", "ms", "check");
  std::printf("%-34s %7s %10.3f  %s\n", "CPU sequential (both)", "-", reference.ms, "reference");
  std::printf("%-34s %7u %10.3f  %s\n",
              "SummaryKernel (7 fused reductions)",
              summary.blocks,
              summary.ms,
              passOrFail(summaryOk));
  std::printf("%-34s %7zu %10.3f  %s\n",
              "PerBikeKernel (segmented)",
              perBike.bikes.size(),
              perBike.ms,
              passOrFail(perBikeOk));
}

static unsigned long long roundedAverageMs(const Totals& totals) {
  const unsigned finished = finishedTrips(totals);
  if (finished == 0) {
    return 0;
  }
  return (totals.rideMs + finished / 2) / finished;
}

static double percentOf(unsigned part, unsigned whole) {
  return whole > 0 ? 100.0 * part / whole : 0.0;
}

static void printDuration(const char* label, unsigned long long ms) {
  std::printf(
      "  %-26s %llu min %02llu s\n", label, ms / MINUTE_MS, ms / SECOND_MS % SECONDS_PER_MINUTE);
}

static void printBusiestAndMostRepairedBike(const std::vector<BikeTotals>& bikes) {
  if (bikes.empty()) {
    return;
  }
  size_t busiest = 0;
  size_t mostRepaired = 0;
  for (size_t b = 1; b < bikes.size(); ++b) {
    if (bikes[b].rides > bikes[busiest].rides) {
      busiest = b;
    }
    if (bikes[b].repairs > bikes[mostRepaired].repairs) {
      mostRepaired = b;
    }
  }
  std::printf("  %-26s b%zu, %u rides\n", "busiest bike:", busiest, bikes[busiest].rides);
  std::printf("  %-26s b%zu, %u trips ended in the workshop\n",
              "most repairs:",
              mostRepaired,
              bikes[mostRepaired].repairs);
}

static void printDayReport(size_t tripCount,
                           const Totals& totals,
                           const std::vector<BikeTotals>& bikes) {
  std::printf("\nBike-share day report: %u stations, %zu bikes\n", STATION_COUNT, bikes.size());
  std::printf("  %-26s %zu\n", "trips started:", tripCount);
  std::printf("  %-26s %u\n", "returned to a dock:", totals.returned);
  std::printf("  %-26s %u\n", "bikes sent to repair:", totals.sentToRepair);
  std::printf("  %-26s %u\n", "still riding at closing:", totals.stillRiding);
  std::printf("  %-26s %u (%.2f%% of finished trips)\n",
              "late returns (> 45 min):",
              totals.late,
              percentOf(totals.late, finishedTrips(totals)));
  std::printf("  %-26s Bs %llu.%02llu\n",
              "fare collected:",
              totals.fareCents / CENTS_PER_BOLIVIANO,
              totals.fareCents % CENTS_PER_BOLIVIANO);
  printDuration("average ride:", roundedAverageMs(totals));
  printDuration("longest ride:", static_cast<unsigned long long>(totals.maxRideMs));
  printBusiestAndMostRepairedBike(bikes);
}

int main() {
  std::printf("Bike-share day report as reductions (chapter 10 use case), backend: %s\n",
              CUDA_BACKEND);
  const unsigned bikes = SYNTHETIC_BIKES;
  const Trips trips = syntheticTrips(SYNTHETIC_TRIPS, bikes, SYNTHETIC_SEED);
  std::printf("source: synthetic, %zu trips, %u bikes (seed 2025)\n", trips.status.size(), bikes);

  const TripsByBike grouped = groupByBike(trips, bikes);
  const CpuReference reference = computeCpuReference(trips, bikes);
  const SummaryRun summary = runSummaryKernel(trips);
  const PerBikeRun perBike = runPerBikeKernel(grouped, bikes);

  const bool summaryOk = summary.totals == reference.totals;
  const bool perBikeOk = perBike.bikes == reference.bikes;
  const Totals& totals = summary.totals;
  const bool everyTripAccountedFor =
      finishedTrips(totals) + totals.stillRiding == trips.status.size();
  const bool bikeRidesAddUp = ridesOfAllBikes(perBike.bikes) == finishedTrips(totals);
  const bool bikeRepairsAddUp = repairsOfAllBikes(perBike.bikes) == totals.sentToRepair;

  printTimingTable(reference, summary, summaryOk, perBike, perBikeOk);
  std::printf("started == returned + repair + riding: %s\n", yesOrNo(everyTripAccountedFor));
  std::printf("sum of rides per bike == finished:     %s\n", yesOrNo(bikeRidesAddUp));
  std::printf("sum of repairs per bike == repair:     %s\n", yesOrNo(bikeRepairsAddUp));
  printDayReport(trips.status.size(), totals, perBike.bikes);

  const bool ok =
      summaryOk && perBikeOk && everyTripAccountedFor && bikeRidesAddUp && bikeRepairsAddUp;
  std::printf("\n%s\n", ok ? "ALL PASS" : "SOME FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_daily_summary.cu -o bikeshare_daily_summary && ./bikeshare_daily_summary

## `bikeshare_kpis.cu`

In [ ]:
%%writefile bikeshare_kpis.cu
#include "cuda_compat.h"

#include <algorithm>
#include <climits>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstring>
#include <limits>
#include <vector>

constexpr unsigned BLOCK_DIM = 256;
constexpr unsigned WARP_SIZE = 32;
constexpr unsigned WARPS_PER_BLOCK = BLOCK_DIM / WARP_SIZE;
constexpr unsigned FULL_WARP_MASK = 0xffffffffu;

constexpr int MINUTE_MS = 60000;
constexpr int SECOND_MS = 1000;
constexpr int SECONDS_PER_MINUTE = 60;
constexpr int INCLUDED_RIDE_MS = 45 * MINUTE_MS;
constexpr int MS_PER_METRE_AT_15_KMH = 240;
constexpr int CENTS_PER_BOLIVIANO = 100;

enum class TripStatus : int { Completed = 0, StillRiding = 1, FalseStart = 2 };

#ifndef CPU_TRIPS
#define CPU_TRIPS (1u << 17)
#endif
const unsigned TRIP_COUNT = BY_BACKEND(1u << 24, CPU_TRIPS);
const int TIMED_REPETITIONS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);
const bool IS_CPU_EMULATION = BY_BACKEND(false, true);

constexpr unsigned PARTIALS_COARSE_FACTOR = 64;
constexpr unsigned SMALL_ATOMICS_COARSE_FACTOR = 128;
constexpr unsigned LARGE_ATOMICS_COARSE_FACTOR = 256;
constexpr double FLOAT_REVENUE_TOLERANCE = 1e-4;

struct Kpi {
  unsigned trips;
  unsigned late;
  unsigned long long fareCents;
  unsigned long long distanceM;
  int minMs;
  int maxMs;
  float fareBs;
};

__host__ __device__ inline Kpi kpiIdentity() {
  return Kpi{0u, 0u, 0ull, 0ull, INT_MAX, INT_MIN, 0.0f};
}

__host__ __device__ inline Kpi combine(const Kpi& a, const Kpi& b) {
  Kpi r;
  r.trips = a.trips + b.trips;
  r.late = a.late + b.late;
  r.fareCents = a.fareCents + b.fareCents;
  r.distanceM = a.distanceM + b.distanceM;
  r.minMs = imin(a.minMs, b.minMs);
  r.maxMs = imax(a.maxMs, b.maxMs);
  r.fareBs = a.fareBs + b.fareBs;
  return r;
}

__host__ __device__ inline Kpi kpiOfTrip(TripStatus status, int ms, int distM, int fareCents) {
  if (status != TripStatus::Completed) {
    return kpiIdentity();
  }
  const unsigned late = ms > INCLUDED_RIDE_MS ? 1u : 0u;
  const float fareBs = static_cast<float>(fareCents) / static_cast<float>(CENTS_PER_BOLIVIANO);
  return Kpi{1u,
             late,
             static_cast<unsigned long long>(fareCents),
             static_cast<unsigned long long>(distM),
             ms,
             ms,
             fareBs};
}

__device__ inline Kpi shuffleDownKpi(const Kpi& v, unsigned delta) {
  Kpi r;
  r.trips = __shfl_down_sync(FULL_WARP_MASK, v.trips, delta);
  r.late = __shfl_down_sync(FULL_WARP_MASK, v.late, delta);
  r.fareCents = __shfl_down_sync(FULL_WARP_MASK, v.fareCents, delta);
  r.distanceM = __shfl_down_sync(FULL_WARP_MASK, v.distanceM, delta);
  r.minMs = __shfl_down_sync(FULL_WARP_MASK, v.minMs, delta);
  r.maxMs = __shfl_down_sync(FULL_WARP_MASK, v.maxMs, delta);
  r.fareBs = __shfl_down_sync(FULL_WARP_MASK, v.fareBs, delta);
  return r;
}

__device__ inline Kpi warpReduceKpi(Kpi v) {
  for (unsigned stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    v = combine(v, shuffleDownKpi(v, stride));
  }
  return v;
}

__device__ inline Kpi blockReduceKpi(Kpi v) {
  __shared__ Kpi warpKpi_s[WARPS_PER_BLOCK];
  const unsigned warp = threadIdx.x / WARP_SIZE;
  const unsigned lane = threadIdx.x % WARP_SIZE;

  v = warpReduceKpi(v);
  if (lane == 0) {
    warpKpi_s[warp] = v;
  }
  __syncthreads();

  Kpi blockKpi = kpiIdentity();
  if (warp == 0) {
    const bool laneHasWarpResult = lane < WARPS_PER_BLOCK;
    if (laneHasWarpResult) {
      blockKpi = warpKpi_s[lane];
    }
    blockKpi = warpReduceKpi(blockKpi);
  }
  return blockKpi;
}

__device__ inline void combineIntoTotalWithAtomics(Kpi* total, const Kpi& blockKpi) {
  atomicAdd(&total->trips, blockKpi.trips);
  atomicAdd(&total->late, blockKpi.late);
  atomicAdd(&total->fareCents, blockKpi.fareCents);
  atomicAdd(&total->distanceM, blockKpi.distanceM);
  atomicMin(&total->minMs, blockKpi.minMs);
  atomicMax(&total->maxMs, blockKpi.maxMs);
  atomicAdd(&total->fareBs, blockKpi.fareBs);
}

template <unsigned COARSE_FACTOR>
__global__ void KpiKernel(const TripStatus* status,
                          const int* rideMs,
                          const int* distanceM,
                          const int* fareCents,
                          unsigned n,
                          Kpi* total,
                          Kpi* partials) {
  Kpi partial = kpiIdentity();
  unsigned segment = COARSE_FACTOR * BLOCK_DIM * blockIdx.x;
  for (unsigned tile = 0; tile < COARSE_FACTOR; ++tile) {
    unsigned i = segment + threadIdx.x + tile * BLOCK_DIM;
    if (i < n) {
      partial = combine(partial, kpiOfTrip(status[i], rideMs[i], distanceM[i], fareCents[i]));
    }
  }

  const Kpi blockKpi = blockReduceKpi(partial);

  const bool combineWithAtomics = partials == nullptr;
  if (threadIdx.x == 0) {
    if (combineWithAtomics) {
      combineIntoTotalWithAtomics(total, blockKpi);
    } else {
      partials[blockIdx.x] = blockKpi;
    }
  }
}

__global__ void KpiCombineKernel(const Kpi* partials, unsigned count, Kpi* out) {
  Kpi partial = kpiIdentity();
  for (unsigned j = threadIdx.x; j < count; j += BLOCK_DIM) {
    partial = combine(partial, partials[j]);
  }
  const Kpi total = blockReduceKpi(partial);
  if (threadIdx.x == 0) {
    *out = total;
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_GAMMA;
    uint64_t mixed = state_;
    mixed = (mixed ^ (mixed >> 30)) * FIRST_MULTIPLIER;
    mixed = (mixed ^ (mixed >> 27)) * SECOND_MULTIPLIER;
    return mixed ^ (mixed >> 31);
  }

  int range(int lowest, int highest) {
    const uint64_t choices = static_cast<uint64_t>(highest - lowest + 1);
    return lowest + static_cast<int>(next() % choices);
  }

 private:
  static constexpr uint64_t GOLDEN_GAMMA = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

constexpr uint64_t DATA_SEED = 2025;
constexpr uint64_t REPLAY_SEED = 7;
constexpr int REPLAYED_BLOCK_SEQUENCES = 1000;

constexpr int PER_MILLE = 1000;
constexpr int WHOLE_PERCENT = 100;
constexpr int STILL_RIDING_PER_MILLE = 30;
constexpr int FALSE_START_PER_MILLE = 50;
constexpr int SHORTEST_TRIP_M = 300;
constexpr int LONGEST_TRIP_M = 8000;
constexpr int LEISURE_RIDE_PERCENT = 4;
constexpr int MEMBER_PERCENT = 60;
constexpr int MEMBER_DISCOUNT_PERCENT = 15;
constexpr int FARE_STEP_MS = 10 * MINUTE_MS;
constexpr int BASE_FARE_CENTS = 300;
constexpr int CENTS_PER_FARE_STEP = 100;
constexpr int METRES_PER_KM = 1000;

struct Trips {
  std::vector<TripStatus> status;
  std::vector<int> rideMs;
  std::vector<int> distanceM;
  std::vector<int> fareCents;
};

struct DeviceTrips {
  TripStatus* status;
  int* rideMs;
  int* distanceM;
  int* fareCents;
};

struct CpuReference {
  Kpi kpi;
  double fareBsInDouble;
  double ms;
};

struct PartialsRun {
  Kpi total;
  std::vector<Kpi> partials;
  unsigned blocks;
  double ms;
};

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

static int fareCentsForRide(int rideMs, bool isMember) {
  const int startedFareSteps = (rideMs + FARE_STEP_MS - 1) / FARE_STEP_MS;
  const int fullFareCents = BASE_FARE_CENTS + startedFareSteps * CENTS_PER_FARE_STEP;
  if (!isMember) {
    return fullFareCents;
  }
  return fullFareCents * (WHOLE_PERCENT - MEMBER_DISCOUNT_PERCENT) / WHOLE_PERCENT;
}

static int syntheticRideMs(SplitMix64& random, int distanceM) {
  const int pedallingMs = distanceM * MS_PER_METRE_AT_15_KMH;
  const int trafficLightsMs = random.range(1 * MINUTE_MS, 12 * MINUTE_MS);
  int rideMs = pedallingMs + trafficLightsMs;

  const bool isLeisureRide = random.range(0, 99) < LEISURE_RIDE_PERCENT;
  if (isLeisureRide) {
    rideMs += random.range(20 * MINUTE_MS, 60 * MINUTE_MS);
  }
  return rideMs;
}

static void appendTrip(Trips& trips, TripStatus status, int rideMs, int distanceM, int fareCents) {
  trips.status.push_back(status);
  trips.rideMs.push_back(rideMs);
  trips.distanceM.push_back(distanceM);
  trips.fareCents.push_back(fareCents);
}

static Trips syntheticDay(unsigned count, uint64_t seed) {
  SplitMix64 random(seed);
  Trips trips;
  for (unsigned i = 0; i < count; ++i) {
    const int outcome = random.range(0, PER_MILLE - 1);
    if (outcome < STILL_RIDING_PER_MILLE) {
      appendTrip(trips, TripStatus::StillRiding, 0, 0, 0);
      continue;
    }
    if (outcome < STILL_RIDING_PER_MILLE + FALSE_START_PER_MILLE) {
      appendTrip(trips, TripStatus::FalseStart, 0, 0, 0);
      continue;
    }

    const int distanceM = random.range(SHORTEST_TRIP_M, LONGEST_TRIP_M);
    const int rideMs = syntheticRideMs(random, distanceM);
    const bool isMember = random.range(0, 99) < MEMBER_PERCENT;
    appendTrip(trips, TripStatus::Completed, rideMs, distanceM, fareCentsForRide(rideMs, isMember));
  }
  return trips;
}

static bool sameIntegers(const Kpi& a, const Kpi& b) {
  return a.trips == b.trips && a.late == b.late && a.fareCents == b.fareCents &&
         a.distanceM == b.distanceM && a.minMs == b.minMs && a.maxMs == b.maxMs;
}

static bool sameFloat(float a, float b) {
  return std::memcmp(&a, &b, sizeof(float)) == 0;
}

static double exactRevenueBs(const Kpi& kpi) {
  return static_cast<double>(kpi.fareCents) / CENTS_PER_BOLIVIANO;
}

static CpuReference computeCpuReference(const Trips& trips) {
  Timer timer;
  CpuReference reference;
  reference.kpi = kpiIdentity();
  reference.fareBsInDouble = 0;
  for (size_t i = 0; i < trips.status.size(); ++i) {
    const Kpi trip =
        kpiOfTrip(trips.status[i], trips.rideMs[i], trips.distanceM[i], trips.fareCents[i]);
    reference.kpi = combine(reference.kpi, trip);
    reference.fareBsInDouble += static_cast<double>(trip.fareBs);
  }
  reference.ms = timer.ms();
  return reference;
}

template <class T>
static T* deviceCopyOf(const std::vector<T>& values) {
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(T)));
  CHECK(cudaMemcpy(pointer, values.data(), values.size() * sizeof(T), cudaMemcpyHostToDevice));
  return pointer;
}

static DeviceTrips copyTripsToDevice(const Trips& trips) {
  DeviceTrips device;
  device.status = deviceCopyOf(trips.status);
  device.rideMs = deviceCopyOf(trips.rideMs);
  device.distanceM = deviceCopyOf(trips.distanceM);
  device.fareCents = deviceCopyOf(trips.fareCents);
  return device;
}

static void freeDeviceTrips(const DeviceTrips& device) {
  CHECK(cudaFree(device.status));
  CHECK(cudaFree(device.rideMs));
  CHECK(cudaFree(device.distanceM));
  CHECK(cudaFree(device.fareCents));
}

static Kpi readKpi(const Kpi* deviceKpi) {
  Kpi kpi;
  CHECK(cudaMemcpy(&kpi, deviceKpi, sizeof(Kpi), cudaMemcpyDeviceToHost));
  return kpi;
}

static void printTableHeader(const CpuReference& reference) {
  const double exactBs = exactRevenueBs(reference.kpi);
  const double sequentialFloatBs = static_cast<double>(reference.kpi.fareBs);
  std::printf("\n%-40s %6s %9s %18s %11s  %s\n",
              "version",
              "blocks",
              "ms",
              "revenue float (Bs)",
              "- exact",
              "integer KPIs");
  std::printf("%-40s %6s %9.3f %18.4f %11.4f  %s\n",
              "CPU sequential loop, float",
              "-",
              reference.ms,
              sequentialFloatBs,
              sequentialFloatBs - exactBs,
              "reference");
  std::printf("%-40s %6s %9s %18.4f %11.4f  %s\n",
              "CPU sequential loop, double",
              "-",
              "-",
              reference.fareBsInDouble,
              reference.fareBsInDouble - exactBs,
              "-");
}

static bool reportRow(const char* name,
                      unsigned blocks,
                      double ms,
                      const Kpi& result,
                      const CpuReference& reference,
                      const char* note) {
  const double exactBs = exactRevenueBs(reference.kpi);
  const double floatBs = static_cast<double>(result.fareBs);
  const bool integersOk = sameIntegers(result, reference.kpi);
  const double relativeError = std::fabs(floatBs - exactBs) / exactBs;
  const bool floatOk = relativeError <= FLOAT_REVENUE_TOLERANCE;
  const bool ok = integersOk && floatOk;
  std::printf("%-40s %6u %9.3f %18.4f %11.4f  %s%s\n",
              name,
              blocks,
              ms,
              floatBs,
              floatBs - exactBs,
              passOrFail(ok),
              note);
  return ok;
}

template <class Work>
static double millisecondsPerRun(Work runOnce) {
  if (WARM_UP_BEFORE_TIMING) {
    runOnce();
  }
  Timer timer;
  for (int repetition = 0; repetition < TIMED_REPETITIONS; ++repetition) {
    runOnce();
  }
  const double milliseconds = timer.ms() / TIMED_REPETITIONS;
  CHECK(cudaGetLastError());
  return milliseconds;
}

static const char* identicalOrDiffer(bool identical) {
  return identical ? "identical" : "DIFFER";
}

template <unsigned COARSE_FACTOR>
static bool checkAtomicsVersion(const char* name,
                                const DeviceTrips& device,
                                unsigned n,
                                const CpuReference& reference) {
  const unsigned blocks = cdiv(n, COARSE_FACTOR * BLOCK_DIM);
  Kpi* total = nullptr;
  CHECK(cudaMalloc(&total, sizeof(Kpi)));
  Kpi* const combineWithAtomicsInsteadOfPartials = nullptr;
  const Kpi identity = kpiIdentity();

  std::vector<Kpi> runs;
  const double ms = millisecondsPerRun([&] {
    CHECK(cudaMemcpy(total, &identity, sizeof(Kpi), cudaMemcpyHostToDevice));
    launch(KpiKernel<COARSE_FACTOR>,
           dim3(blocks),
           dim3(BLOCK_DIM),
           device.status,
           device.rideMs,
           device.distanceM,
           device.fareCents,
           n,
           total,
           combineWithAtomicsInsteadOfPartials);
    runs.push_back(readKpi(total));
  });
  CHECK(cudaFree(total));

  bool floatStable = true;
  bool integersStable = true;
  for (const Kpi& run : runs) {
    floatStable = floatStable && sameFloat(run.fareBs, runs[0].fareBs);
    integersStable = integersStable && sameIntegers(run, runs[0]);
  }
  char note[96];
  std::snprintf(note,
                sizeof note,
                "  (%zu run%s: float %s, integers %s)",
                runs.size(),
                runs.size() == 1 ? "" : "s",
                identicalOrDiffer(floatStable),
                identicalOrDiffer(integersStable));
  const bool rowOk = reportRow(name, blocks, ms, runs.back(), reference, note);
  return rowOk && integersStable;
}

static PartialsRun runPartialsVersion(const DeviceTrips& device, unsigned n) {
  PartialsRun run;
  run.blocks = cdiv(n, PARTIALS_COARSE_FACTOR * BLOCK_DIM);
  Kpi* partials = nullptr;
  Kpi* total = nullptr;
  CHECK(cudaMalloc(&partials, run.blocks * sizeof(Kpi)));
  CHECK(cudaMalloc(&total, sizeof(Kpi)));
  Kpi* const unusedAtomicTotal = nullptr;

  Timer timer;
  launch(KpiKernel<PARTIALS_COARSE_FACTOR>,
         dim3(run.blocks),
         dim3(BLOCK_DIM),
         device.status,
         device.rideMs,
         device.distanceM,
         device.fareCents,
         n,
         unusedAtomicTotal,
         partials);
  launch(KpiCombineKernel, dim3(1), dim3(BLOCK_DIM), partials, run.blocks, total);
  run.ms = timer.ms();
  CHECK(cudaGetLastError());

  run.total = readKpi(total);
  run.partials.resize(run.blocks);
  CHECK(
      cudaMemcpy(run.partials.data(), partials, run.blocks * sizeof(Kpi), cudaMemcpyDeviceToHost));
  CHECK(cudaFree(partials));
  CHECK(cudaFree(total));
  return run;
}

static void shuffleInPlace(std::vector<unsigned>& sequence, SplitMix64& random) {
  for (size_t j = sequence.size() - 1; j > 0; --j) {
    const size_t other = static_cast<size_t>(random.next() % (j + 1));
    std::swap(sequence[j], sequence[other]);
  }
}

template <class T>
static size_t sortAndCountDistinct(std::vector<T>& values) {
  std::sort(values.begin(), values.end());
  return static_cast<size_t>(std::unique(values.begin(), values.end()) - values.begin());
}

static bool checkBlockSequenceReplay(const std::vector<Kpi>& partials,
                                     const CpuReference& reference) {
  std::vector<unsigned> finishSequence(partials.size());
  for (unsigned j = 0; j < finishSequence.size(); ++j) {
    finishSequence[j] = j;
  }
  std::vector<float> floatTotals;
  std::vector<unsigned long long> centTotals;
  SplitMix64 random(REPLAY_SEED);
  for (int replay = 0; replay < REPLAYED_BLOCK_SEQUENCES; ++replay) {
    shuffleInPlace(finishSequence, random);
    float floatTotal = 0.0f;
    unsigned long long centTotal = 0;
    for (unsigned block : finishSequence) {
      floatTotal += partials[block].fareBs;
      centTotal += partials[block].fareCents;
    }
    floatTotals.push_back(floatTotal);
    centTotals.push_back(centTotal);
  }

  const size_t distinctFloatTotals = sortAndCountDistinct(floatTotals);
  const size_t distinctCentTotals = sortAndCountDistinct(centTotals);
  std::printf(
      "\n%zu block results added in 1000 random sequences (what atomicAdd sees on a GPU):\n",
      partials.size());
  std::printf("  float bolivianos: %zu distinct totals, from %.4f to %.4f\n",
              distinctFloatTotals,
              static_cast<double>(floatTotals.front()),
              static_cast<double>(floatTotals[distinctFloatTotals - 1]));
  std::printf("  integer centavos: %zu distinct total%s, %llu.%02llu\n",
              distinctCentTotals,
              distinctCentTotals == 1 ? "" : "s",
              centTotals.front() / CENTS_PER_BOLIVIANO,
              centTotals.front() % CENTS_PER_BOLIVIANO);
  return distinctCentTotals == 1 && centTotals.front() == reference.kpi.fareCents;
}

static double percentOf(unsigned part, unsigned whole) {
  return whole > 0 ? 100.0 * part / whole : 0.0;
}

static double averageOf(unsigned long long total, unsigned count) {
  return count > 0 ? static_cast<double>(total) / count : 0.0;
}

static double floatStepAt(double value) {
  const float asFloat = static_cast<float>(value);
  const float next = std::nextafter(asFloat, std::numeric_limits<float>::infinity());
  return static_cast<double>(next) - static_cast<double>(asFloat);
}

static void printKpis(const Kpi& kpi) {
  std::printf("\nday KPIs: %u completed trips, revenue Bs %llu.%02llu, %u late returns (%.2f%%)\n",
              kpi.trips,
              kpi.fareCents / CENTS_PER_BOLIVIANO,
              kpi.fareCents % CENTS_PER_BOLIVIANO,
              kpi.late,
              percentOf(kpi.late, kpi.trips));
  std::printf(
      "          shortest ride %d min %02d s, longest %d min %02d s, %.1f km ridden, %.0f m "
      "per trip\n",
      kpi.minMs / MINUTE_MS,
      kpi.minMs / SECOND_MS % SECONDS_PER_MINUTE,
      kpi.maxMs / MINUTE_MS,
      kpi.maxMs / SECOND_MS % SECONDS_PER_MINUTE,
      static_cast<double>(kpi.distanceM) / METRES_PER_KM,
      averageOf(kpi.distanceM, kpi.trips));
  const double exactBs = exactRevenueBs(kpi);
  std::printf(
      "float revenue: one rounding per addition to 24 bits; at %.0f Bs one float step is %.4f "
      "Bs.\n",
      exactBs,
      floatStepAt(exactBs));
}

int main() {
  std::printf(
      "Bike-share KPIs in one pass, a struct reduction (chapter 10 use case), backend: %s\n",
      CUDA_BACKEND);
  const unsigned n = TRIP_COUNT;
  const Trips trips = syntheticDay(n, DATA_SEED);
  std::printf("source: synthetic, %u trips (seed 2025), %u threads per block\n", n, BLOCK_DIM);

  const CpuReference reference = computeCpuReference(trips);
  const DeviceTrips device = copyTripsToDevice(trips);
  printTableHeader(reference);

  const bool smallFactorOk = checkAtomicsVersion<SMALL_ATOMICS_COARSE_FACTOR>(
      "KpiKernel CF=128, atomics per field", device, n, reference);
  const bool largeFactorOk = checkAtomicsVersion<LARGE_ATOMICS_COARSE_FACTOR>(
      "KpiKernel CF=256, atomics per field", device, n, reference);

  const PartialsRun partialsRun = runPartialsVersion(device, n);
  const bool partialsOk = reportRow("KpiKernel CF=64, partials + 2nd pass",
                                    partialsRun.blocks + 1,
                                    partialsRun.ms,
                                    partialsRun.total,
                                    reference,
                                    "  (fixed sequence)");
  std::printf("%-40s %6s %9s %18.4f %11.4f  %s\n",
              "exact: integer centavos / 100",
              "-",
              "-",
              exactRevenueBs(reference.kpi),
              0.0,
              "the right answer");

  const bool replayOk = checkBlockSequenceReplay(partialsRun.partials, reference);
  printKpis(partialsRun.total);
  freeDeviceTrips(device);

  const bool ok = smallFactorOk && largeFactorOk && partialsOk && replayOk;
  std::printf("\n%s\n", ok ? "ALL PASS" : "SOME FAILED");
  if (IS_CPU_EMULATION) {
    std::printf(
        "(CPU emulation: blocks run one after another, so the atomics always see the same "
        "sequence.)\n");
  }
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_kpis.cu -o bikeshare_kpis && ./bikeshare_kpis

## `bikeshare_nearest_truck.cu`

In [ ]:
%%writefile bikeshare_nearest_truck.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <vector>

constexpr unsigned BLOCK_DIM = 256;
constexpr unsigned COARSE_FACTOR = 8;

constexpr int MS_PER_METRE_AT_25_KMH = 144;
constexpr int STREET_DETOUR_PERCENT = 125;
constexpr int MS_PER_METRE = MS_PER_METRE_AT_25_KMH * STREET_DETOUR_PERCENT / 100;
constexpr int TRUCK_CAPACITY_BIKES = 20;

constexpr unsigned NO_ETA = 0xffffffffu;
constexpr unsigned long long NO_KEY = ~0ull;
constexpr int TRUCK_INDEX_BITS = 32;
constexpr unsigned long long TRUCK_INDEX_MASK = (1ull << TRUCK_INDEX_BITS) - 1;

enum class TruckState : int { Parked = 0, Driving = 1, Loading = 2, OffDuty = 3 };

struct EmptyStation {
  int id;
  int x;
  int y;
  int docks;
  int curbOpensInMs;
};

__host__ __device__ inline int iabs(int x) {
  return x < 0 ? -x : x;
}

__host__ __device__ inline unsigned long long umin64(unsigned long long a, unsigned long long b) {
  return a < b ? a : b;
}

__host__ __device__ inline int manhattanMetres(int fromX, int fromY, int toX, int toY) {
  return iabs(fromX - toX) + iabs(fromY - toY);
}

__host__ __device__ inline bool canRestock(TruckState state, int bikesLoaded) {
  return state != TruckState::OffDuty && bikesLoaded > 0;
}

__host__ __device__ inline unsigned etaMs(int truckX,
                                          int truckY,
                                          TruckState state,
                                          int bikesLoaded,
                                          int busyMs,
                                          const EmptyStation& station) {
  if (!canRestock(state, bikesLoaded)) {
    return NO_ETA;
  }
  const int driveMs = manhattanMetres(truckX, truckY, station.x, station.y) * MS_PER_METRE;
  const int arrivalMs = busyMs + driveMs;
  const int stopMs = arrivalMs > station.curbOpensInMs ? arrivalMs : station.curbOpensInMs;
  return static_cast<unsigned>(stopMs);
}

__host__ __device__ inline unsigned long long packKey(unsigned eta, unsigned truck) {
  return (static_cast<unsigned long long>(eta) << TRUCK_INDEX_BITS) | truck;
}

__host__ __device__ inline unsigned long long candidateKey(unsigned eta, unsigned truck) {
  return eta == NO_ETA ? NO_KEY : packKey(eta, truck);
}

static unsigned etaOfKey(unsigned long long key) {
  return static_cast<unsigned>(key >> TRUCK_INDEX_BITS);
}

static unsigned truckOfKey(unsigned long long key) {
  return static_cast<unsigned>(key & TRUCK_INDEX_MASK);
}

__global__ void NearestTruckKernel(const int* truckX,
                                   const int* truckY,
                                   const TruckState* state,
                                   const int* bikesLoaded,
                                   const int* busyMs,
                                   unsigned numTrucks,
                                   const EmptyStation* stations,
                                   unsigned long long* best) {
  __shared__ unsigned long long key_s[BLOCK_DIM];
  const EmptyStation station = stations[blockIdx.y];
  unsigned t = threadIdx.x;
  unsigned segment = COARSE_FACTOR * blockDim.x * blockIdx.x;

  unsigned long long key = NO_KEY;
  for (unsigned tile = 0; tile < COARSE_FACTOR; ++tile) {
    unsigned truck = segment + t + tile * BLOCK_DIM;
    if (truck < numTrucks) {
      unsigned eta = etaMs(
          truckX[truck], truckY[truck], state[truck], bikesLoaded[truck], busyMs[truck], station);
      key = umin64(key, candidateKey(eta, truck));
    }
  }
  key_s[t] = key;

  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      key_s[t] = umin64(key_s[t], key_s[t + stride]);
    }
  }

  if (t == 0 && key_s[0] != NO_KEY) {
    atomicMin(&best[blockIdx.y], key_s[0]);
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_GAMMA;
    uint64_t mixed = state_;
    mixed = (mixed ^ (mixed >> 30)) * FIRST_MULTIPLIER;
    mixed = (mixed ^ (mixed >> 27)) * SECOND_MULTIPLIER;
    return mixed ^ (mixed >> 31);
  }

  int range(int lowest, int highest) {
    const uint64_t choices = static_cast<uint64_t>(highest - lowest + 1);
    return lowest + static_cast<int>(next() % choices);
  }

 private:
  static constexpr uint64_t GOLDEN_GAMMA = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MULTIPLIER = 0x94D049BB133111EBull;

  uint64_t state_;
};

constexpr uint64_t DATA_SEED = 99;
constexpr unsigned DEFAULT_TRUCKS = BY_BACKEND(1u << 18, 1u << 14);
const int TIMED_REPETITIONS = BY_BACKEND(10, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

constexpr int MINUTE_MS = 60000;
constexpr int SECOND_MS = 1000;
constexpr int SECONDS_PER_MINUTE = 60;
constexpr int CITY_SIZE_M = 6000;
constexpr int STATION_COUNT = 48;
constexpr int FEWEST_DOCKS = 12;
constexpr int MOST_DOCKS = 30;
constexpr int EMPTY_STATION_PERCENT = 25;
constexpr int CURB_CLOSED_PERCENT = 50;
constexpr int LONGEST_CURB_CLOSURE_MS = 6 * MINUTE_MS;
constexpr int PARKED_PERCENT = 35;
constexpr int DRIVING_PERCENT = 30;
constexpr int LOADING_PERCENT = 20;

struct Station {
  int id;
  int x;
  int y;
  int docks;
  int bikesDocked;
};

struct Trucks {
  std::vector<int> x;
  std::vector<int> y;
  std::vector<TruckState> state;
  std::vector<int> bikesLoaded;
  std::vector<int> busyMs;
};

struct DeviceTrucks {
  int* x;
  int* y;
  TruckState* state;
  int* bikesLoaded;
  int* busyMs;
};

struct CpuReference {
  std::vector<unsigned long long> best;
  double ms;
};

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

static void appendTruck(Trucks& trucks, TruckState state, int bikesLoaded, int busyMs) {
  trucks.state.push_back(state);
  trucks.bikesLoaded.push_back(bikesLoaded);
  trucks.busyMs.push_back(busyMs);
}

static void addRandomTruck(Trucks& trucks, SplitMix64& random) {
  trucks.x.push_back(random.range(0, CITY_SIZE_M));
  trucks.y.push_back(random.range(0, CITY_SIZE_M));
  const int roll = random.range(0, 99);
  if (roll < PARKED_PERCENT) {
    appendTruck(trucks, TruckState::Parked, random.range(0, TRUCK_CAPACITY_BIKES), 0);
  } else if (roll < PARKED_PERCENT + DRIVING_PERCENT) {
    const int bikesLoaded = random.range(1, TRUCK_CAPACITY_BIKES);
    appendTruck(trucks, TruckState::Driving, bikesLoaded, random.range(0, 10 * MINUTE_MS));
  } else if (roll < PARKED_PERCENT + DRIVING_PERCENT + LOADING_PERCENT) {
    const int bikesLoaded = random.range(0, TRUCK_CAPACITY_BIKES - 1);
    appendTruck(trucks, TruckState::Loading, bikesLoaded, random.range(0, 15 * MINUTE_MS));
  } else {
    appendTruck(trucks, TruckState::OffDuty, 0, 0);
  }
}

static Trucks makeTrucks(unsigned count, SplitMix64& random) {
  Trucks trucks;
  for (unsigned k = 0; k < count; ++k) {
    addRandomTruck(trucks, random);
  }
  return trucks;
}

static std::vector<Station> makeStations(SplitMix64& random) {
  std::vector<Station> stations(STATION_COUNT);
  for (int s = 0; s < STATION_COUNT; ++s) {
    Station& station = stations[s];
    station.id = s;
    station.x = random.range(0, CITY_SIZE_M);
    station.y = random.range(0, CITY_SIZE_M);
    station.docks = random.range(FEWEST_DOCKS, MOST_DOCKS);
    const bool emptiedByRushHour = random.range(0, 99) < EMPTY_STATION_PERCENT;
    station.bikesDocked = emptiedByRushHour ? 0 : random.range(1, station.docks);
  }
  return stations;
}

static std::vector<EmptyStation> emptyStationsOf(const std::vector<Station>& stations,
                                                 SplitMix64& random) {
  std::vector<EmptyStation> empty;
  for (const Station& station : stations) {
    if (station.bikesDocked > 0) {
      continue;
    }
    const bool curbClosedForRushHour = random.range(0, 99) < CURB_CLOSED_PERCENT;
    const int curbOpensInMs = curbClosedForRushHour ? random.range(1, LONGEST_CURB_CLOSURE_MS) : 0;
    empty.push_back(EmptyStation{station.id, station.x, station.y, station.docks, curbOpensInMs});
  }
  return empty;
}

static unsigned truckEta(const Trucks& trucks, unsigned truck, const EmptyStation& station) {
  return etaMs(trucks.x[truck],
               trucks.y[truck],
               trucks.state[truck],
               trucks.bikesLoaded[truck],
               trucks.busyMs[truck],
               station);
}

static CpuReference computeCpuReference(const Trucks& trucks,
                                        const std::vector<EmptyStation>& empty) {
  const unsigned truckCount = static_cast<unsigned>(trucks.x.size());
  CpuReference reference;
  reference.best.assign(empty.size(), NO_KEY);
  Timer timer;
  for (size_t s = 0; s < empty.size(); ++s) {
    for (unsigned k = 0; k < truckCount; ++k) {
      reference.best[s] = umin64(reference.best[s], candidateKey(truckEta(trucks, k, empty[s]), k));
    }
  }
  reference.ms = timer.ms();
  return reference;
}

static std::vector<unsigned> trucksSharingBestEta(const Trucks& trucks,
                                                  const std::vector<EmptyStation>& empty,
                                                  const std::vector<unsigned long long>& best) {
  const unsigned truckCount = static_cast<unsigned>(trucks.x.size());
  std::vector<unsigned> ties(empty.size(), 0);
  for (size_t s = 0; s < empty.size(); ++s) {
    if (best[s] == NO_KEY) {
      continue;
    }
    for (unsigned k = 0; k < truckCount; ++k) {
      if (truckEta(trucks, k, empty[s]) == etaOfKey(best[s])) {
        ++ties[s];
      }
    }
  }
  return ties;
}

template <class T>
static T* deviceCopyOf(const std::vector<T>& values) {
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, values.size() * sizeof(T)));
  CHECK(cudaMemcpy(pointer, values.data(), values.size() * sizeof(T), cudaMemcpyHostToDevice));
  return pointer;
}

static DeviceTrucks copyTrucksToDevice(const Trucks& trucks) {
  DeviceTrucks device;
  device.x = deviceCopyOf(trucks.x);
  device.y = deviceCopyOf(trucks.y);
  device.state = deviceCopyOf(trucks.state);
  device.bikesLoaded = deviceCopyOf(trucks.bikesLoaded);
  device.busyMs = deviceCopyOf(trucks.busyMs);
  return device;
}

static void freeDeviceTrucks(const DeviceTrucks& device) {
  CHECK(cudaFree(device.x));
  CHECK(cudaFree(device.y));
  CHECK(cudaFree(device.state));
  CHECK(cudaFree(device.bikesLoaded));
  CHECK(cudaFree(device.busyMs));
}

template <class Work>
static double millisecondsPerRun(Work runOnce) {
  if (WARM_UP_BEFORE_TIMING) {
    runOnce();
  }
  Timer timer;
  for (int repetition = 0; repetition < TIMED_REPETITIONS; ++repetition) {
    runOnce();
  }
  const double milliseconds = timer.ms() / TIMED_REPETITIONS;
  CHECK(cudaGetLastError());
  return milliseconds;
}

static double runNearestTruckKernel(const Trucks& trucks,
                                    const std::vector<EmptyStation>& empty,
                                    const dim3& grid,
                                    std::vector<unsigned long long>& best) {
  const unsigned truckCount = static_cast<unsigned>(trucks.x.size());
  const DeviceTrucks deviceTrucks = copyTrucksToDevice(trucks);
  EmptyStation* deviceStations = deviceCopyOf(empty);
  const std::vector<unsigned long long> noKeys(empty.size(), NO_KEY);
  unsigned long long* deviceBest = deviceCopyOf(noKeys);
  const size_t bestBytes = empty.size() * sizeof(unsigned long long);

  const double ms = millisecondsPerRun([&] {
    CHECK(cudaMemcpy(deviceBest, noKeys.data(), bestBytes, cudaMemcpyHostToDevice));
    launch(NearestTruckKernel,
           grid,
           dim3(BLOCK_DIM),
           deviceTrucks.x,
           deviceTrucks.y,
           deviceTrucks.state,
           deviceTrucks.bikesLoaded,
           deviceTrucks.busyMs,
           truckCount,
           deviceStations,
           deviceBest);
  });
  best.resize(empty.size());
  CHECK(cudaMemcpy(best.data(), deviceBest, bestBytes, cudaMemcpyDeviceToHost));

  freeDeviceTrucks(deviceTrucks);
  CHECK(cudaFree(deviceStations));
  CHECK(cudaFree(deviceBest));
  return ms;
}

static bool parseArguments(int argc, char** argv, unsigned& trucks) {
  trucks = DEFAULT_TRUCKS;
  if (argc > 1) {
    trucks = static_cast<unsigned>(std::strtoul(argv[1], nullptr, 10));
  }
  return trucks > 0;
}

static void printTimingTable(double cpuMs, const dim3& grid, double gpuMs, bool ok) {
  std::printf("\n%-36s %13s %10s  %s\n", "version", "grid", "ms", "check");
  std::printf("%-36s %13s %10.3f  %s\n", "CPU sequential argmin", "-", cpuMs, "reference");
  char gridText[32];
  std::snprintf(gridText, sizeof gridText, "%u x %u", grid.x, grid.y);
  std::printf("%-36s %13s %10.3f  %s\n",
              "NearestTruckKernel (packed atomicMin)",
              gridText,
              gpuMs,
              passOrFail(ok));
}

static unsigned countSharedChoices(std::vector<unsigned> chosen) {
  std::sort(chosen.begin(), chosen.end());
  unsigned conflicts = 0;
  for (size_t k = 1; k < chosen.size(); ++k) {
    if (chosen[k] == chosen[k - 1]) {
      ++conflicts;
    }
  }
  return conflicts;
}

static void printPlan(const Trucks& trucks,
                      const std::vector<EmptyStation>& empty,
                      const std::vector<unsigned long long>& best,
                      const std::vector<unsigned>& ties) {
  std::printf("\nempty station  docks  curb opens  nearest truck  bikes  ETA (min:s)  same ETA\n");
  unsigned withoutTruck = 0;
  std::vector<unsigned> chosen;
  for (size_t s = 0; s < empty.size(); ++s) {
    const EmptyStation& station = empty[s];
    if (best[s] == NO_KEY) {
      ++withoutTruck;
      continue;
    }
    const unsigned eta = etaOfKey(best[s]);
    const unsigned truck = truckOfKey(best[s]);
    chosen.push_back(truck);
    std::printf("s%-12d  %5d  %7d:%02d  t%-12u  %5d  %8u:%02u  %8u\n",
                station.id,
                station.docks,
                station.curbOpensInMs / MINUTE_MS,
                station.curbOpensInMs / SECOND_MS % SECONDS_PER_MINUTE,
                truck,
                trucks.bikesLoaded[truck],
                eta / MINUTE_MS,
                eta / SECOND_MS % SECONDS_PER_MINUTE,
                ties[s]);
  }

  std::printf("empty stations without any truck able to restock them: %u\n", withoutTruck);
  std::printf("empty stations whose nearest truck is also the nearest truck of another: %u\n",
              countSharedChoices(chosen));
  std::printf(
      "(each argmin is independent; a real rebalancing plan sends one truck, updates its load\n"
      " and position, and recomputes, or resolves the conflicts afterwards)\n");
}

int main(int argc, char** argv) {
  unsigned truckCount = 0;
  if (!parseArguments(argc, argv, truckCount)) {
    std::fprintf(stderr, "usage: %s [trucks >= 1]\n", argv[0]);
    return 1;
  }
  std::printf("Nearest rebalancing truck per empty station (chapter 10 use case), backend: %s\n",
              CUDA_BACKEND);

  SplitMix64 random(DATA_SEED);
  const std::vector<Station> stations = makeStations(random);
  const std::vector<EmptyStation> empty = emptyStationsOf(stations, random);
  const Trucks trucks = makeTrucks(truckCount, random);
  std::printf("%u trucks x %zu empty stations (of %d) = %llu ETA estimates (seed 99)\n",
              truckCount,
              empty.size(),
              STATION_COUNT,
              static_cast<unsigned long long>(truckCount) * empty.size());
  if (empty.empty()) {
    std::printf("\nno empty station to restock\n\nALL PASS\n");
    return 0;
  }

  const CpuReference reference = computeCpuReference(trucks, empty);
  const std::vector<unsigned> ties = trucksSharingBestEta(trucks, empty, reference.best);

  const dim3 grid(cdiv(truckCount, COARSE_FACTOR * BLOCK_DIM), static_cast<unsigned>(empty.size()));
  std::vector<unsigned long long> best;
  const double gpuMs = runNearestTruckKernel(trucks, empty, grid, best);
  const bool ok = best == reference.best;

  printTimingTable(reference.ms, grid, gpuMs, ok);
  printPlan(trucks, empty, best, ties);

  std::printf("\n%s\n", ok ? "ALL PASS" : "SOME FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_nearest_truck.cu -o bikeshare_nearest_truck && ./bikeshare_nearest_truck

## `reduction.cu`

In [ ]:
%%writefile reduction.cu
#include "cuda_compat.h"

#include <climits>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <vector>

constexpr unsigned SINGLE_BLOCK_THREADS = 1024;
constexpr unsigned SINGLE_BLOCK_INPUTS = 2 * SINGLE_BLOCK_THREADS;
constexpr unsigned BLOCK_DIM = 256;
constexpr unsigned WARP_SIZE = 32;
constexpr unsigned WARPS_IN_SINGLE_BLOCK = SINGLE_BLOCK_THREADS / WARP_SIZE;

constexpr float SUM_IDENTITY = 0.0f;
constexpr int MAX_IDENTITY = INT_MIN;

#ifndef CPU_N
#define CPU_N (1u << 18)
#endif
const unsigned LARGE_INPUT_LENGTH = BY_BACKEND(1u << 25, CPU_N);
const int TIMED_REPETITIONS = BY_BACKEND(20, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);

constexpr unsigned SMALL_COARSE_FACTOR = 4;
constexpr unsigned LARGE_COARSE_FACTOR = 32;
constexpr unsigned MAX_COARSE_FACTOR = 8;

constexpr double SINGLE_BLOCK_TOLERANCE = 1e-5;
constexpr double MULTI_BLOCK_TOLERANCE = 1e-4;

constexpr uint64_t DATA_SEED = 10;
constexpr uint64_t RANDOM_INTEGER_LIMIT = 1000000000u;
constexpr int PLANTED_MAXIMUM = 1999999999;

__global__ void SimpleSumReductionKernel(float* input, float* output) {
  unsigned i = 2 * threadIdx.x;
  for (unsigned stride = 1; stride <= blockDim.x; stride *= 2) {
    if (threadIdx.x % stride == 0) {
      input[i] += input[i + stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__global__ void ConvergentSumReductionKernel(float* input, float* output) {
  unsigned i = threadIdx.x;
  for (unsigned stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x < stride) {
      input[i] += input[i + stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__global__ void SharedMemorySumReductionKernel(const float* input, float* output) {
  __shared__ float input_s[SINGLE_BLOCK_THREADS];
  unsigned t = threadIdx.x;
  input_s[t] = input[t] + input[t + SINGLE_BLOCK_THREADS];
  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      input_s[t] += input_s[t + stride];
    }
  }
  if (t == 0) {
    *output = input_s[0];
  }
}

__device__ inline float valueOrSumIdentity(const float* input, unsigned index, unsigned n) {
  return index < n ? input[index] : SUM_IDENTITY;
}

__device__ inline int valueOrMaxIdentity(const int* input, unsigned index, unsigned n) {
  return index < n ? input[index] : MAX_IDENTITY;
}

__global__ void SegmentedSumReductionKernel(const float* input, float* output, unsigned n) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned segment = 2 * blockDim.x * blockIdx.x;
  unsigned i = segment + threadIdx.x;
  unsigned t = threadIdx.x;
  float firstHalf = valueOrSumIdentity(input, i, n);
  float secondHalf = valueOrSumIdentity(input, i + BLOCK_DIM, n);
  input_s[t] = firstHalf + secondHalf;

  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      input_s[t] += input_s[t + stride];
    }
  }

  if (t == 0) {
    atomicAdd(output, input_s[0]);
  }
}

template <unsigned COARSE_FACTOR>
__global__ void CoarsenedSumReductionKernel(const float* input, float* output, unsigned n) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned i = segment + threadIdx.x;
  unsigned t = threadIdx.x;
  float partialSum = SUM_IDENTITY;
  for (unsigned tile = 0; tile < COARSE_FACTOR * 2; ++tile) {
    partialSum += valueOrSumIdentity(input, i + tile * BLOCK_DIM, n);
  }
  input_s[t] = partialSum;

  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      input_s[t] += input_s[t + stride];
    }
  }

  if (t == 0) {
    atomicAdd(output, input_s[0]);
  }
}

template <unsigned COARSE_FACTOR>
__global__ void CoarsenedMaxReductionKernel(const int* input, int* output, unsigned n) {
  __shared__ int input_s[BLOCK_DIM];
  unsigned segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned i = segment + threadIdx.x;
  unsigned t = threadIdx.x;
  int partialMax = MAX_IDENTITY;
  for (unsigned tile = 0; tile < COARSE_FACTOR * 2; ++tile) {
    partialMax = imax(partialMax, valueOrMaxIdentity(input, i + tile * BLOCK_DIM, n));
  }
  input_s[t] = partialMax;

  for (unsigned stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (t < stride) {
      input_s[t] = imax(input_s[t], input_s[t + stride]);
    }
  }

  if (t == 0) {
    atomicMax(output, input_s[0]);
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_GAMMA;
    uint64_t mixed = state_;
    mixed = (mixed ^ (mixed >> 30)) * FIRST_MULTIPLIER;
    mixed = (mixed ^ (mixed >> 27)) * SECOND_MULTIPLIER;
    return mixed ^ (mixed >> 31);
  }

  float uniformBelowOne() {
    const uint64_t mantissa = next() >> (64 - FLOAT_MANTISSA_BITS);
    return static_cast<float>(mantissa * (1.0 / (1u << FLOAT_MANTISSA_BITS)));
  }

 private:
  static constexpr uint64_t GOLDEN_GAMMA = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MULTIPLIER = 0x94D049BB133111EBull;
  static constexpr int FLOAT_MANTISSA_BITS = 24;

  uint64_t state_;
};

struct InputData {
  std::vector<float> small;
  std::vector<float> large;
  std::vector<int> integers;
};

struct CpuReferences {
  double smallSum = 0;
  double largeSum = 0;
  double largeSumMs = 0;
  float sequentialFloatSum = 0;
  double sequentialFloatMs = 0;
  int maximum = MAX_IDENTITY;
  double maximumMs = 0;
};

struct DeviceBuffers {
  float* small;
  float* large;
  float* sumOutput;
  int* integers;
  int* maxOutput;
};

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

template <class T>
static T* allocateOnDevice(size_t count) {
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(T)));
  return pointer;
}

template <class T>
static void copyToDevice(T* destination, const std::vector<T>& source) {
  CHECK(cudaMemcpy(destination, source.data(), source.size() * sizeof(T), cudaMemcpyHostToDevice));
}

template <class T>
static void setOnDevice(T* destination, T value) {
  CHECK(cudaMemcpy(destination, &value, sizeof(T), cudaMemcpyHostToDevice));
}

template <class T>
static T readFromDevice(const T* source) {
  T value{};
  CHECK(cudaMemcpy(&value, source, sizeof(T), cudaMemcpyDeviceToHost));
  return value;
}

template <class Work>
static double millisecondsPerRun(Work runOnce) {
  if (WARM_UP_BEFORE_TIMING) {
    runOnce();
  }
  Timer timer;
  for (int repetition = 0; repetition < TIMED_REPETITIONS; ++repetition) {
    runOnce();
  }
  const double milliseconds = timer.ms() / TIMED_REPETITIONS;
  CHECK(cudaGetLastError());
  return milliseconds;
}

static InputData makeInputData() {
  InputData data;
  data.small.resize(SINGLE_BLOCK_INPUTS);
  data.large.resize(LARGE_INPUT_LENGTH);
  data.integers.resize(LARGE_INPUT_LENGTH);

  SplitMix64 random(DATA_SEED);
  for (float& value : data.small) {
    value = random.uniformBelowOne();
  }
  for (float& value : data.large) {
    value = random.uniformBelowOne();
  }
  for (int& value : data.integers) {
    value = static_cast<int>(random.next() % RANDOM_INTEGER_LIMIT);
  }
  data.integers[LARGE_INPUT_LENGTH / 3] = PLANTED_MAXIMUM;
  return data;
}

static CpuReferences computeCpuReferences(const InputData& data) {
  CpuReferences references;
  for (float value : data.small) {
    references.smallSum += value;
  }

  Timer doubleTimer;
  for (float value : data.large) {
    references.largeSum += value;
  }
  references.largeSumMs = doubleTimer.ms();

  Timer floatTimer;
  for (float value : data.large) {
    references.sequentialFloatSum += value;
  }
  references.sequentialFloatMs = floatTimer.ms();

  Timer maximumTimer;
  for (int value : data.integers) {
    references.maximum = value > references.maximum ? value : references.maximum;
  }
  references.maximumMs = maximumTimer.ms();
  return references;
}

static DeviceBuffers allocateDeviceBuffers(const InputData& data) {
  DeviceBuffers device;
  device.small = allocateOnDevice<float>(SINGLE_BLOCK_INPUTS);
  device.large = allocateOnDevice<float>(LARGE_INPUT_LENGTH);
  device.sumOutput = allocateOnDevice<float>(1);
  device.integers = allocateOnDevice<int>(LARGE_INPUT_LENGTH);
  device.maxOutput = allocateOnDevice<int>(1);
  copyToDevice(device.large, data.large);
  copyToDevice(device.integers, data.integers);
  return device;
}

static void freeDeviceBuffers(const DeviceBuffers& device) {
  CHECK(cudaFree(device.small));
  CHECK(cudaFree(device.large));
  CHECK(cudaFree(device.sumOutput));
  CHECK(cudaFree(device.integers));
  CHECK(cudaFree(device.maxOutput));
}

static void printTableHeader() {
  std::printf("\n%-30s %10s %7s %10s %18s %10s  %s\n",
              "kernel",
              "n",
              "blocks",
              "ms",
              "result",
              "rel.err",
              "check");
}

static void printUntimedReferenceRow(const char* name, unsigned n, double value) {
  std::printf("%-30s %10u %7s %10s %18.4f %10s  %s\n", name, n, "-", "-", value, "-", "reference");
}

static void printTimedReferenceRow(const char* name, unsigned n, double ms, double value) {
  std::printf("%-30s %10u %7s %10.3f %18.4f %10s  %s\n", name, n, "-", ms, value, "-", "reference");
}

static bool printSumRow(const char* name,
                        unsigned n,
                        unsigned blocks,
                        double ms,
                        double value,
                        double reference,
                        double tolerance) {
  const double relativeError = std::fabs(value - reference) / std::fabs(reference);
  const bool ok = relativeError <= tolerance;
  std::printf("%-30s %10u %7u %10.3f %18.4f %10.2e  %s\n",
              name,
              n,
              blocks,
              ms,
              value,
              relativeError,
              passOrFail(ok));
  return ok;
}

template <class Kernel>
static bool checkInPlaceKernel(const char* name,
                               Kernel kernel,
                               const InputData& data,
                               const DeviceBuffers& device,
                               double reference) {
  const double ms = millisecondsPerRun([&] {
    copyToDevice(device.small, data.small);
    launch(kernel, dim3(1), dim3(SINGLE_BLOCK_THREADS), device.small, device.sumOutput);
  });
  const float result = readFromDevice(device.sumOutput);
  return printSumRow(name, SINGLE_BLOCK_INPUTS, 1, ms, result, reference, SINGLE_BLOCK_TOLERANCE);
}

static bool checkSharedMemoryKernel(const InputData& data,
                                    const DeviceBuffers& device,
                                    double reference) {
  copyToDevice(device.small, data.small);
  const float* input = device.small;
  const double ms = millisecondsPerRun([&] {
    launch(SharedMemorySumReductionKernel,
           dim3(1),
           dim3(SINGLE_BLOCK_THREADS),
           input,
           device.sumOutput);
  });
  const float result = readFromDevice(device.sumOutput);
  return printSumRow(
      "3 shared memory", SINGLE_BLOCK_INPUTS, 1, ms, result, reference, SINGLE_BLOCK_TOLERANCE);
}

static bool checkSingleBlockKernels(const InputData& data,
                                    const DeviceBuffers& device,
                                    const CpuReferences& references) {
  printUntimedReferenceRow("CPU double (small input)", SINGLE_BLOCK_INPUTS, references.smallSum);
  const bool simpleOk = checkInPlaceKernel(
      "1 simple (in place)", SimpleSumReductionKernel, data, device, references.smallSum);
  const bool convergentOk = checkInPlaceKernel(
      "2 convergent (in place)", ConvergentSumReductionKernel, data, device, references.smallSum);
  const bool sharedMemoryOk = checkSharedMemoryKernel(data, device, references.smallSum);
  return simpleOk && convergentOk && sharedMemoryOk;
}

template <class Kernel>
static bool checkMultiBlockSumKernel(const char* name,
                                     Kernel kernel,
                                     unsigned elementsPerBlock,
                                     const DeviceBuffers& device,
                                     double reference) {
  const unsigned blocks = cdiv(LARGE_INPUT_LENGTH, elementsPerBlock);
  const float* input = device.large;
  const double ms = millisecondsPerRun([&] {
    setOnDevice(device.sumOutput, SUM_IDENTITY);
    launch(kernel, dim3(blocks), dim3(BLOCK_DIM), input, device.sumOutput, LARGE_INPUT_LENGTH);
  });
  const float result = readFromDevice(device.sumOutput);
  return printSumRow(
      name, LARGE_INPUT_LENGTH, blocks, ms, result, reference, MULTI_BLOCK_TOLERANCE);
}

static void printLargeInputReferences(const CpuReferences& references) {
  printTimedReferenceRow(
      "CPU double (large input)", LARGE_INPUT_LENGTH, references.largeSumMs, references.largeSum);
  const double floatSum = static_cast<double>(references.sequentialFloatSum);
  const double floatRelativeError = std::fabs(floatSum - references.largeSum) / references.largeSum;
  std::printf("%-30s %10u %7s %10.3f %18.4f %10.2e  %s\n",
              "CPU float, sequential",
              LARGE_INPUT_LENGTH,
              "-",
              references.sequentialFloatMs,
              floatSum,
              floatRelativeError,
              "(for comparison)");
}

static bool checkMultiBlockSumKernels(const DeviceBuffers& device,
                                      const CpuReferences& references) {
  printLargeInputReferences(references);
  const bool segmentedOk = checkMultiBlockSumKernel("4 segmented + atomicAdd",
                                                    SegmentedSumReductionKernel,
                                                    2 * BLOCK_DIM,
                                                    device,
                                                    references.largeSum);
  const bool smallFactorOk =
      checkMultiBlockSumKernel("5 coarsened, factor 4",
                               CoarsenedSumReductionKernel<SMALL_COARSE_FACTOR>,
                               SMALL_COARSE_FACTOR * 2 * BLOCK_DIM,
                               device,
                               references.largeSum);
  const bool largeFactorOk =
      checkMultiBlockSumKernel("5 coarsened, factor 32",
                               CoarsenedSumReductionKernel<LARGE_COARSE_FACTOR>,
                               LARGE_COARSE_FACTOR * 2 * BLOCK_DIM,
                               device,
                               references.largeSum);
  return segmentedOk && smallFactorOk && largeFactorOk;
}

static bool checkMaxKernel(const DeviceBuffers& device, const CpuReferences& references) {
  const unsigned blocks = cdiv(LARGE_INPUT_LENGTH, MAX_COARSE_FACTOR * 2 * BLOCK_DIM);
  const int* input = device.integers;
  const double ms = millisecondsPerRun([&] {
    setOnDevice(device.maxOutput, MAX_IDENTITY);
    launch(CoarsenedMaxReductionKernel<MAX_COARSE_FACTOR>,
           dim3(blocks),
           dim3(BLOCK_DIM),
           input,
           device.maxOutput,
           LARGE_INPUT_LENGTH);
  });
  const int result = readFromDevice(device.maxOutput);
  const bool ok = result == references.maximum;

  std::printf("%-30s %10u %7s %10.3f %18d %10s  %s\n",
              "CPU max, sequential",
              LARGE_INPUT_LENGTH,
              "-",
              references.maximumMs,
              references.maximum,
              "-",
              "reference");
  std::printf("%-30s %10u %7u %10.3f %18d %10s  %s\n",
              "6 max, coarsened factor 8",
              LARGE_INPUT_LENGTH,
              blocks,
              ms,
              result,
              "exact",
              passOrFail(ok));
  return ok;
}

struct StepActivity {
  unsigned activeThreads = 0;
  unsigned busyWarps = 0;
};

static bool simpleKernelThreadIsActive(unsigned thread, unsigned stride) {
  return thread % stride == 0;
}

static bool convergentKernelThreadIsActive(unsigned thread, unsigned stride) {
  return thread < stride;
}

template <class IsActive>
static StepActivity countStepActivity(IsActive isActive, unsigned stride) {
  StepActivity activity;
  for (unsigned warp = 0; warp < WARPS_IN_SINGLE_BLOCK; ++warp) {
    unsigned activeLanes = 0;
    for (unsigned lane = 0; lane < WARP_SIZE; ++lane) {
      if (isActive(warp * WARP_SIZE + lane, stride)) {
        ++activeLanes;
      }
    }
    activity.activeThreads += activeLanes;
    if (activeLanes > 0) {
      ++activity.busyWarps;
    }
  }
  return activity;
}

static double laneUtilisationPercent(unsigned long long additions, unsigned long long warpSteps) {
  return 100.0 * additions / (warpSteps * static_cast<double>(WARP_SIZE));
}

static void printDivergenceAnalysis() {
  std::printf("\nDivergence analysis, one block of %u threads = %u warps, %u inputs\n",
              SINGLE_BLOCK_THREADS,
              WARPS_IN_SINGLE_BLOCK,
              SINGLE_BLOCK_INPUTS);
  std::printf(
      "%6s | %-26s | %-26s\n", "step", "simple: threads / warps", "convergent: threads / warps");

  unsigned long long additions = 0;
  unsigned long long simpleWarpSteps = 0;
  unsigned long long convergentWarpSteps = 0;
  for (unsigned step = 0; (1u << step) <= SINGLE_BLOCK_THREADS; ++step) {
    const unsigned simpleStride = 1u << step;
    const unsigned convergentStride = SINGLE_BLOCK_THREADS >> step;
    const StepActivity simple = countStepActivity(simpleKernelThreadIsActive, simpleStride);
    const StepActivity convergent =
        countStepActivity(convergentKernelThreadIsActive, convergentStride);

    additions += simple.activeThreads;
    simpleWarpSteps += simple.busyWarps;
    convergentWarpSteps += convergent.busyWarps;
    std::printf("%6u | %10u / %-13u | %10u / %-13u\n",
                step,
                simple.activeThreads,
                simple.busyWarps,
                convergent.activeThreads,
                convergent.busyWarps);
  }

  std::printf("total additions %llu; warp-steps: simple %llu, convergent %llu\n",
              additions,
              simpleWarpSteps,
              convergentWarpSteps);
  std::printf("lane utilisation (additions / (warp-steps*32)): simple %.1f%%, convergent %.1f%%\n",
              laneUtilisationPercent(additions, simpleWarpSteps),
              laneUtilisationPercent(additions, convergentWarpSteps));
  std::printf(
      "global traffic of kernels 1-2: %llu loads + %llu stores; kernel 3: %u loads + 1 store\n",
      2 * additions,
      additions,
      SINGLE_BLOCK_INPUTS);
}

int main() {
  std::printf("Parallel reduction (chapter 10), backend: %s\n", CUDA_BACKEND);

  const InputData data = makeInputData();
  const CpuReferences references = computeCpuReferences(data);
  const DeviceBuffers device = allocateDeviceBuffers(data);

  printTableHeader();
  const bool singleBlockOk = checkSingleBlockKernels(data, device, references);
  const bool multiBlockOk = checkMultiBlockSumKernels(device, references);
  const bool maxOk = checkMaxKernel(device, references);
  const bool ok = singleBlockOk && multiBlockOk && maxOk;

  printDivergenceAnalysis();

  freeDeviceBuffers(device);
  std::printf("\n%s\n", ok ? "ALL PASS" : "SOME FAILED");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. reduction.cu -o reduction && ./reduction

## `reduction_book.cu`

In [ ]:
%%writefile reduction_book.cu
#include "cuda_compat.h"

#include <climits>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <set>
#include <vector>

#define WARP_SIZE 32
#define BLOCK_DIM 1024
#define COARSE_FACTOR 4
static_assert(BLOCK_DIM == WARP_SIZE * WARP_SIZE, "the book's second stage needs 32 warp sums");

constexpr unsigned int FULL_WARP_MASK = 0xffffffff;
constexpr float SUM_IDENTITY = 0.0f;
constexpr int MAX_IDENTITY = INT_MIN;

constexpr unsigned SINGLE_BLOCK_INPUTS = 2 * BLOCK_DIM;
constexpr unsigned SEGMENT_LENGTH = 2 * BLOCK_DIM;
constexpr unsigned COARSENED_SEGMENT_LENGTH = COARSE_FACTOR * 2 * BLOCK_DIM;

#ifndef CPU_N
#define CPU_N (1u << 16)
#endif
const unsigned LARGE_INPUT_LENGTH = BY_BACKEND(1u << 25, CPU_N);
const int TIMED_REPETITIONS = BY_BACKEND(20, 1);
const bool WARM_UP_BEFORE_TIMING = BY_BACKEND(true, false);
const bool IS_CPU_EMULATION = BY_BACKEND(false, true);
static_assert(CPU_N % COARSENED_SEGMENT_LENGTH == 0, "CPU_N must be a multiple of 8192");
static_assert(CPU_N >= 2 * COARSENED_SEGMENT_LENGTH, "CPU_N must be at least 16384");

__global__ void SimpleSumReductionKernel(float* input, float* output) {
  unsigned int i = 2 * threadIdx.x;
  for (unsigned int stride = 1; stride <= blockDim.x; stride *= 2) {
    if (threadIdx.x % stride == 0) {
      input[i] += input[i + stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__global__ void ConvergentSumReductionKernel(float* input, float* output) {
  unsigned int i = threadIdx.x;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x < stride) {
      input[i] += input[i + stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__global__ void SharedMemorySumReductionKernel(float* input, float* output) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned int t = threadIdx.x;
  input_s[t] = input[t] + input[t + BLOCK_DIM];
  for (unsigned int stride = blockDim.x / 2; stride >= 1; stride /= 2) {
    __syncthreads();
    if (threadIdx.x < stride) {
      input_s[t] += input_s[t + stride];
    }
  }
  if (threadIdx.x == 0) {
    *output = input_s[0];
  }
}

__device__ unsigned int warpIdx() {
  return threadIdx.x / WARP_SIZE;
}

__device__ unsigned int laneIdx() {
  return threadIdx.x % WARP_SIZE;
}

__device__ __inline__ float warp_reduce(float val) {
  float partialSum = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialSum += __shfl_down_sync(0xffffffff, partialSum, stride);
  }
  return partialSum;
}

__global__ void WarpLevelSumReductionKernel(float* input, float* output) {
  __shared__ float input_s[BLOCK_DIM];
  unsigned int t = threadIdx.x;
  input_s[t] = input[t] + input[t + BLOCK_DIM];
  for (unsigned int stride = blockDim.x / 2; stride >= WARP_SIZE; stride /= 2) {
    __syncthreads();
    if (threadIdx.x < stride) {
      input_s[t] += input_s[t + stride];
    }
  }
  __syncthreads();
  if (warpIdx() == 0) {
    float partialSum = input_s[t];
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      *output = partialSum;
    }
  }
}

__global__ void TwoStageWarpLevelSumReductionKernel(float* input, float* output) {
  unsigned int t = threadIdx.x;
  float partialSum = input[t] + input[t + BLOCK_DIM];
  partialSum = warp_reduce(partialSum);

  __shared__ float partialSums_s[BLOCK_DIM / WARP_SIZE];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float partialSum = partialSums_s[t];
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      *output = partialSum;
    }
  }
}

__global__ void reduce_kernel(float* input, float* output, unsigned int N) {
  (void)N;

  unsigned int segment = 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = input[i] + input[i + BLOCK_DIM];
  partialSum = warp_reduce(partialSum);

  __shared__ float partialSums_s[BLOCK_DIM / WARP_SIZE];
  if (threadIdx.x % WARP_SIZE == 0) {
    partialSums_s[threadIdx.x / WARP_SIZE] = partialSum;
  }
  __syncthreads();

  if (threadIdx.x < WARP_SIZE) {
    float partialSum = partialSums_s[threadIdx.x];
    partialSum = warp_reduce(partialSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(partialSum, cuda::memory_order_relaxed);
    }
  }
}

__global__ void CoarsenedSumReductionKernel(float* input, float* output) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = input[i];
  for (unsigned int c = 1; c < COARSE_FACTOR * 2; ++c) {
    partialSum += input[i + c * BLOCK_DIM];
  }

  partialSum = warp_reduce(partialSum);
  __shared__ float partialSums_s[BLOCK_DIM / WARP_SIZE];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float blockSum = warp_reduce(partialSums_s[laneIdx()]);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(blockSum, cuda::memory_order_relaxed);
    }
  }
}

__global__ void ReversedConvergentSumReductionKernel(float* input, float* output) {
  unsigned int i = blockDim.x + threadIdx.x;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x >= blockDim.x - stride) {
      input[i] += input[i - stride];
    }
    __syncthreads();
  }
  if (threadIdx.x == blockDim.x - 1) {
    *output = input[2 * blockDim.x - 1];
  }
}

__device__ bool laneHasWarpResult() {
  return laneIdx() < BLOCK_DIM / WARP_SIZE;
}

__device__ __inline__ int warp_reduce_max(int val) {
  int partialMax = val;
  for (unsigned int stride = WARP_SIZE / 2; stride > 0; stride /= 2) {
    partialMax = imax(partialMax, __shfl_down_sync(FULL_WARP_MASK, partialMax, stride));
  }
  return partialMax;
}

__global__ void CoarsenedMaxReductionKernel(int* input, int* output) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  int partialMax = input[i];
  for (unsigned int c = 1; c < COARSE_FACTOR * 2; ++c) {
    partialMax = imax(partialMax, input[i + c * BLOCK_DIM]);
  }

  partialMax = warp_reduce_max(partialMax);
  __shared__ int partialMaxs_s[BLOCK_DIM / WARP_SIZE];
  if (laneIdx() == 0) {
    partialMaxs_s[warpIdx()] = partialMax;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    int warpMax = laneHasWarpResult() ? partialMaxs_s[laneIdx()] : MAX_IDENTITY;
    int blockMax = warp_reduce_max(warpMax);
    if (threadIdx.x == 0) {
      atomicMax(output, blockMax);
    }
  }
}

__global__ void CoarsenedSumReductionKernelN(float* input, float* output, unsigned int N) {
  unsigned int segment = COARSE_FACTOR * 2 * blockDim.x * blockIdx.x;
  unsigned int i = segment + threadIdx.x;
  float partialSum = SUM_IDENTITY;
  for (unsigned int c = 0; c < COARSE_FACTOR * 2; ++c) {
    if (i + c * BLOCK_DIM < N) {
      partialSum += input[i + c * BLOCK_DIM];
    }
  }

  partialSum = warp_reduce(partialSum);
  __shared__ float partialSums_s[BLOCK_DIM / WARP_SIZE];
  if (laneIdx() == 0) {
    partialSums_s[warpIdx()] = partialSum;
  }
  __syncthreads();

  if (warpIdx() == 0) {
    float warpSum = laneHasWarpResult() ? partialSums_s[laneIdx()] : SUM_IDENTITY;
    float blockSum = warp_reduce(warpSum);
    if (threadIdx.x == 0) {
      cuda::atomic_ref<float, cuda::thread_scope_device> output_ref(*output);
      output_ref.fetch_add(blockSum, cuda::memory_order_relaxed);
    }
  }
}

__device__ void copyOwnedAndNeverWrittenToTrace(const float* input,
                                                float* traceRow,
                                                unsigned int ownedPosition,
                                                unsigned int neverWrittenPosition) {
  traceRow[ownedPosition] = input[ownedPosition];
  traceRow[neverWrittenPosition] = input[neverWrittenPosition];
}

__global__ void SimpleSumReductionKernelTraced(float* input, float* output, float* trace) {
  unsigned int i = 2 * threadIdx.x;
  unsigned int step = 0;
  for (unsigned int stride = 1; stride <= blockDim.x; stride *= 2) {
    if (threadIdx.x % stride == 0) {
      input[i] += input[i + stride];
    }
    __syncthreads();
    copyOwnedAndNeverWrittenToTrace(input, trace + step * 2 * blockDim.x, i, i + 1);
    ++step;
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__global__ void ConvergentSumReductionKernelTraced(float* input, float* output, float* trace) {
  unsigned int i = threadIdx.x;
  unsigned int step = 0;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x < stride) {
      input[i] += input[i + stride];
    }
    __syncthreads();
    copyOwnedAndNeverWrittenToTrace(input, trace + step * 2 * blockDim.x, i, i + blockDim.x);
    ++step;
  }
  if (threadIdx.x == 0) {
    *output = input[0];
  }
}

__global__ void ReversedConvergentSumReductionKernelTraced(float* input,
                                                           float* output,
                                                           float* trace) {
  unsigned int i = blockDim.x + threadIdx.x;
  unsigned int step = 0;
  for (unsigned int stride = blockDim.x; stride >= 1; stride /= 2) {
    if (threadIdx.x >= blockDim.x - stride) {
      input[i] += input[i - stride];
    }
    __syncthreads();
    copyOwnedAndNeverWrittenToTrace(input, trace + step * 2 * blockDim.x, i, threadIdx.x);
    ++step;
  }
  if (threadIdx.x == blockDim.x - 1) {
    *output = input[2 * blockDim.x - 1];
  }
}

constexpr double SINGLE_BLOCK_TOLERANCE = 1e-5;
constexpr double MULTI_BLOCK_TOLERANCE = 1e-4;

constexpr uint64_t DATA_SEED = 10;
constexpr int RANDOM_INTEGER_MAGNITUDE = 1000000000;
constexpr int PLANTED_MAXIMUM = 1999999999;

constexpr unsigned MEMORY_SEGMENT_BYTES = 128;
constexpr unsigned BOOK_ANALYSIS_ELEMENTS = 256;
constexpr unsigned BOOK_ANALYSIS_THREADS = BOOK_ANALYSIS_ELEMENTS / 2;
constexpr unsigned BOOK_SIMPLE_RESOURCES = 864;
constexpr unsigned BOOK_CONVERGENT_RESOURCES = 384;
constexpr unsigned BOOK_COMMITTED_RESULTS = 255;
constexpr unsigned BOOK_SIMPLE_REQUESTS = 141;
constexpr unsigned BOOK_CONVERGENT_REQUESTS = 36;
constexpr unsigned BOOK_SHARED_MEMORY_REQUESTS = 9;

constexpr unsigned DIVERGENCE_EXERCISE_ELEMENTS = 1024;
constexpr unsigned DIVERGENCE_EXERCISE_THREADS = DIVERGENCE_EXERCISE_ELEMENTS / 2;
constexpr size_t FIFTH_ITERATION_INDEX = 4;
constexpr unsigned EXPECTED_SIMPLE_DIVERGENT_WARPS = 16;
constexpr unsigned EXPECTED_CONVERGENT_DIVERGENT_WARPS = 0;

constexpr unsigned TRACE_THREADS = 4;
constexpr unsigned TRACE_ELEMENTS = 2 * TRACE_THREADS;
constexpr unsigned TRACE_ITERATIONS = 3;
constexpr float TRACE_INPUT[TRACE_ELEMENTS] = {6, 2, 7, 4, 5, 8, 3, 1};
constexpr float TRACE_TOTAL = 36.0f;

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_GAMMA;
    uint64_t mixed = state_;
    mixed = (mixed ^ (mixed >> 30)) * FIRST_MULTIPLIER;
    mixed = (mixed ^ (mixed >> 27)) * SECOND_MULTIPLIER;
    return mixed ^ (mixed >> 31);
  }

  float uniformBelowOne() {
    const uint64_t mantissa = next() >> (64 - FLOAT_MANTISSA_BITS);
    return static_cast<float>(mantissa * (1.0 / (1u << FLOAT_MANTISSA_BITS)));
  }

 private:
  static constexpr uint64_t GOLDEN_GAMMA = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MULTIPLIER = 0x94D049BB133111EBull;
  static constexpr int FLOAT_MANTISSA_BITS = 24;

  uint64_t state_;
};

struct InputData {
  std::vector<float> small;
  std::vector<float> large;
  std::vector<int> integers;
};

struct DeviceBuffers {
  float* small;
  float* large;
  float* sumOutput;
  int* integers;
  int* maxOutput;
};

enum class InputUse { ReadOnly, OverwrittenInPlace };

static const char* passOrFail(bool ok) {
  return ok ? "PASS" : "FAIL";
}

template <class T>
static T* allocateOnDevice(size_t count) {
  T* pointer = nullptr;
  CHECK(cudaMalloc(&pointer, count * sizeof(T)));
  return pointer;
}

template <class T>
static void copyToDevice(T* destination, const T* source, size_t count) {
  CHECK(cudaMemcpy(destination, source, count * sizeof(T), cudaMemcpyHostToDevice));
}

template <class T>
static void copyToHost(T* destination, const T* source, size_t count) {
  CHECK(cudaMemcpy(destination, source, count * sizeof(T), cudaMemcpyDeviceToHost));
}

template <class T>
static void setOnDevice(T* destination, T value) {
  copyToDevice(destination, &value, 1);
}

template <class T>
static T readFromDevice(const T* source) {
  T value{};
  copyToHost(&value, source, 1);
  return value;
}

template <class Work>
static double millisecondsPerRun(Work runOnce) {
  if (WARM_UP_BEFORE_TIMING) {
    runOnce();
  }
  Timer timer;
  for (int repetition = 0; repetition < TIMED_REPETITIONS; ++repetition) {
    runOnce();
  }
  const double milliseconds = timer.ms() / TIMED_REPETITIONS;
  CHECK(cudaGetLastError());
  return milliseconds;
}

static InputData makeInputData() {
  InputData data;
  data.small.resize(SINGLE_BLOCK_INPUTS);
  data.large.resize(LARGE_INPUT_LENGTH);
  data.integers.resize(LARGE_INPUT_LENGTH);

  SplitMix64 random(DATA_SEED);
  for (float& value : data.small) {
    value = random.uniformBelowOne();
  }
  for (float& value : data.large) {
    value = random.uniformBelowOne();
  }
  const uint64_t signedRangeSize = 2u * RANDOM_INTEGER_MAGNITUDE + 1u;
  for (int& value : data.integers) {
    value = static_cast<int>(random.next() % signedRangeSize) - RANDOM_INTEGER_MAGNITUDE;
  }
  data.integers[LARGE_INPUT_LENGTH / 3 + 5] = PLANTED_MAXIMUM;
  return data;
}

static double sumOfFirst(const std::vector<float>& values, unsigned count) {
  double sum = 0;
  for (unsigned k = 0; k < count; ++k) {
    sum += values[k];
  }
  return sum;
}

static int maximumOf(const std::vector<int>& values) {
  int maximum = MAX_IDENTITY;
  for (int value : values) {
    maximum = value > maximum ? value : maximum;
  }
  return maximum;
}

static DeviceBuffers allocateDeviceBuffers(const InputData& data) {
  DeviceBuffers device;
  device.small = allocateOnDevice<float>(SINGLE_BLOCK_INPUTS);
  device.large = allocateOnDevice<float>(LARGE_INPUT_LENGTH);
  device.sumOutput = allocateOnDevice<float>(1);
  device.integers = allocateOnDevice<int>(LARGE_INPUT_LENGTH);
  device.maxOutput = allocateOnDevice<int>(1);
  copyToDevice(device.large, data.large.data(), LARGE_INPUT_LENGTH);
  copyToDevice(device.integers, data.integers.data(), LARGE_INPUT_LENGTH);
  return device;
}

static void freeDeviceBuffers(const DeviceBuffers& device) {
  CHECK(cudaFree(device.small));
  CHECK(cudaFree(device.large));
  CHECK(cudaFree(device.sumOutput));
  CHECK(cudaFree(device.integers));
  CHECK(cudaFree(device.maxOutput));
}

static void printBanner() {
  std::printf("Reduction as the book writes it (chapter 10), backend: %s\n", CUDA_BACKEND);
  std::printf("BLOCK_DIM %d, WARP_SIZE %d, COARSE_FACTOR %d, %d timed run(s)\n",
              BLOCK_DIM,
              WARP_SIZE,
              COARSE_FACTOR,
              TIMED_REPETITIONS);
}

static void printTableHeader() {
  std::printf("\n%-40s %9s %6s %9s %16s %9s  %s\n",
              "kernel",
              "n",
              "blocks",
              "ms",
              "result",
              "rel.err",
              "check");
}

static void printReferenceRow(unsigned n, double reference) {
  std::printf("%-40s %9u %6s %9s %16.4f %9s  %s\n",
              "CPU, double accumulation",
              n,
              "-",
              "-",
              reference,
              "-",
              "reference");
}

static double relativeError(double value, double reference) {
  if (reference == 0) {
    return std::fabs(value);
  }
  return std::fabs(value - reference) / std::fabs(reference);
}

static bool printSumRow(const char* name,
                        unsigned n,
                        unsigned blocks,
                        double ms,
                        double value,
                        double reference,
                        double tolerance) {
  const double error = relativeError(value, reference);
  const bool ok = error <= tolerance;
  std::printf(
      "%-40s %9u %6u %9.3f %16.4f %9.2e  %s\n", name, n, blocks, ms, value, error, passOrFail(ok));
  return ok;
}

template <class Kernel>
static bool checkSingleBlockKernel(const char* name,
                                   Kernel kernel,
                                   InputUse inputUse,
                                   const InputData& data,
                                   const DeviceBuffers& device,
                                   double reference) {
  const bool overwritesInput = inputUse == InputUse::OverwrittenInPlace;
  if (!overwritesInput) {
    copyToDevice(device.small, data.small.data(), SINGLE_BLOCK_INPUTS);
  }
  const double ms = millisecondsPerRun([&] {
    if (overwritesInput) {
      copyToDevice(device.small, data.small.data(), SINGLE_BLOCK_INPUTS);
    }
    launch(kernel, dim3(1), dim3(BLOCK_DIM), device.small, device.sumOutput);
  });
  const float result = readFromDevice(device.sumOutput);
  return printSumRow(name, SINGLE_BLOCK_INPUTS, 1, ms, result, reference, SINGLE_BLOCK_TOLERANCE);
}

static bool checkSingleBlockKernels(const InputData& data, const DeviceBuffers& device) {
  const double reference = sumOfFirst(data.small, SINGLE_BLOCK_INPUTS);
  printReferenceRow(SINGLE_BLOCK_INPUTS, reference);

  const bool simpleOk = checkSingleBlockKernel("1 SimpleSumReductionKernel",
                                               SimpleSumReductionKernel,
                                               InputUse::OverwrittenInPlace,
                                               data,
                                               device,
                                               reference);
  const bool convergentOk = checkSingleBlockKernel("2 ConvergentSumReductionKernel",
                                                   ConvergentSumReductionKernel,
                                                   InputUse::OverwrittenInPlace,
                                                   data,
                                                   device,
                                                   reference);
  const bool reversedOk = checkSingleBlockKernel("ex3 ReversedConvergentSumReductionKernel",
                                                 ReversedConvergentSumReductionKernel,
                                                 InputUse::OverwrittenInPlace,
                                                 data,
                                                 device,
                                                 reference);
  const bool sharedMemoryOk = checkSingleBlockKernel("3 SharedMemorySumReductionKernel",
                                                     SharedMemorySumReductionKernel,
                                                     InputUse::ReadOnly,
                                                     data,
                                                     device,
                                                     reference);
  const bool warpLevelOk = checkSingleBlockKernel("5 WarpLevelSumReductionKernel",
                                                  WarpLevelSumReductionKernel,
                                                  InputUse::ReadOnly,
                                                  data,
                                                  device,
                                                  reference);
  const bool twoStageOk = checkSingleBlockKernel("6 TwoStageWarpLevelSumReductionKernel",
                                                 TwoStageWarpLevelSumReductionKernel,
                                                 InputUse::ReadOnly,
                                                 data,
                                                 device,
                                                 reference);
  return simpleOk && convergentOk && reversedOk && sharedMemoryOk && warpLevelOk && twoStageOk;
}

static bool checkReduceKernel(const DeviceBuffers& device, double reference) {
  const unsigned blocks = LARGE_INPUT_LENGTH / SEGMENT_LENGTH;
  const double ms = millisecondsPerRun([&] {
    setOnDevice(device.sumOutput, SUM_IDENTITY);
    launch(reduce_kernel,
           dim3(blocks),
           dim3(BLOCK_DIM),
           device.large,
           device.sumOutput,
           LARGE_INPUT_LENGTH);
  });
  const float result = readFromDevice(device.sumOutput);
  return printSumRow("7 reduce_kernel (atomic_ref)",
                     LARGE_INPUT_LENGTH,
                     blocks,
                     ms,
                     result,
                     reference,
                     MULTI_BLOCK_TOLERANCE);
}

static bool checkCoarsenedSumKernel(const DeviceBuffers& device, double reference) {
  const unsigned blocks = LARGE_INPUT_LENGTH / COARSENED_SEGMENT_LENGTH;
  const double ms = millisecondsPerRun([&] {
    setOnDevice(device.sumOutput, SUM_IDENTITY);
    launch(
        CoarsenedSumReductionKernel, dim3(blocks), dim3(BLOCK_DIM), device.large, device.sumOutput);
  });
  const float result = readFromDevice(device.sumOutput);
  return printSumRow("8 CoarsenedSumReductionKernel",
                     LARGE_INPUT_LENGTH,
                     blocks,
                     ms,
                     result,
                     reference,
                     MULTI_BLOCK_TOLERANCE);
}

static bool checkCoarsenedMaxKernel(const InputData& data, const DeviceBuffers& device) {
  const unsigned blocks = LARGE_INPUT_LENGTH / COARSENED_SEGMENT_LENGTH;
  const double ms = millisecondsPerRun([&] {
    setOnDevice(device.maxOutput, MAX_IDENTITY);
    launch(CoarsenedMaxReductionKernel,
           dim3(blocks),
           dim3(BLOCK_DIM),
           device.integers,
           device.maxOutput);
  });
  const int result = readFromDevice(device.maxOutput);
  const bool ok = result == maximumOf(data.integers);
  std::printf("%-40s %9u %6u %9.3f %16d %9s  %s\n",
              "ex4 CoarsenedMaxReductionKernel",
              LARGE_INPUT_LENGTH,
              blocks,
              ms,
              result,
              "exact",
              passOrFail(ok));
  return ok;
}

static bool checkAnyLengthKernel(const InputData& data, const DeviceBuffers& device) {
  const unsigned testLengths[] = {1u,
                                  1000u,
                                  COARSENED_SEGMENT_LENGTH - 1,
                                  COARSENED_SEGMENT_LENGTH + 1,
                                  LARGE_INPUT_LENGTH / 2 + 5u,
                                  LARGE_INPUT_LENGTH - 777u};
  bool ok = true;
  for (unsigned length : testLengths) {
    const unsigned blocks = cdiv(length, COARSENED_SEGMENT_LENGTH);
    const double ms = millisecondsPerRun([&] {
      setOnDevice(device.sumOutput, SUM_IDENTITY);
      launch(CoarsenedSumReductionKernelN,
             dim3(blocks),
             dim3(BLOCK_DIM),
             device.large,
             device.sumOutput,
             length);
    });
    const float result = readFromDevice(device.sumOutput);
    const double reference = sumOfFirst(data.large, length);
    const bool lengthOk = printSumRow("ex5 CoarsenedSumReductionKernelN",
                                      length,
                                      blocks,
                                      ms,
                                      result,
                                      reference,
                                      MULTI_BLOCK_TOLERANCE);
    ok = ok && lengthOk;
  }
  return ok;
}

static bool checkMultiBlockKernels(const InputData& data, const DeviceBuffers& device) {
  const double reference = sumOfFirst(data.large, LARGE_INPUT_LENGTH);
  printReferenceRow(LARGE_INPUT_LENGTH, reference);

  const bool reduceOk = checkReduceKernel(device, reference);
  const bool coarsenedOk = checkCoarsenedSumKernel(device, reference);
  const bool maxOk = checkCoarsenedMaxKernel(data, device);
  const bool anyLengthOk = checkAnyLengthKernel(data, device);
  return reduceOk && coarsenedOk && maxOk && anyLengthOk;
}

enum class Pattern { Simple, Convergent, Reversed };

static std::vector<unsigned> stridesOf(Pattern pattern, unsigned threads) {
  std::vector<unsigned> strides;
  if (pattern == Pattern::Simple) {
    for (unsigned stride = 1; stride <= threads; stride *= 2) {
      strides.push_back(stride);
    }
  } else {
    for (unsigned stride = threads; stride >= 1; stride /= 2) {
      strides.push_back(stride);
    }
  }
  return strides;
}

static bool isActive(Pattern pattern, unsigned thread, unsigned stride, unsigned threads) {
  switch (pattern) {
    case Pattern::Simple:
      return thread % stride == 0;
    case Pattern::Convergent:
      return thread < stride;
    case Pattern::Reversed:
      return thread >= threads - stride;
  }
  return false;
}

static unsigned ownedPosition(Pattern pattern, unsigned thread, unsigned threads) {
  switch (pattern) {
    case Pattern::Simple:
      return 2 * thread;
    case Pattern::Convergent:
      return thread;
    case Pattern::Reversed:
      return threads + thread;
  }
  return 0;
}

static unsigned partnerPosition(Pattern pattern,
                                unsigned thread,
                                unsigned stride,
                                unsigned threads) {
  const unsigned owned = ownedPosition(pattern, thread, threads);
  if (pattern == Pattern::Reversed) {
    return owned - stride;
  }
  return owned + stride;
}

static unsigned memorySegmentOf(unsigned floatPosition) {
  return static_cast<unsigned>(floatPosition * sizeof(float) / MEMORY_SEGMENT_BYTES);
}

struct WarpAccesses {
  unsigned activeLanes = 0;
  std::set<unsigned> ownSegments;
  std::set<unsigned> partnerSegments;
};

static WarpAccesses warpAccesses(Pattern pattern,
                                 unsigned threads,
                                 unsigned stride,
                                 unsigned warp) {
  WarpAccesses accesses;
  for (unsigned lane = 0; lane < WARP_SIZE && warp * WARP_SIZE + lane < threads; ++lane) {
    const unsigned thread = warp * WARP_SIZE + lane;
    if (!isActive(pattern, thread, stride, threads)) {
      continue;
    }
    ++accesses.activeLanes;
    accesses.ownSegments.insert(memorySegmentOf(ownedPosition(pattern, thread, threads)));
    accesses.partnerSegments.insert(
        memorySegmentOf(partnerPosition(pattern, thread, stride, threads)));
  }
  return accesses;
}

struct IterationStats {
  unsigned stride;
  unsigned activeThreads;
  unsigned activeWarps;
  unsigned divergentWarps;
  unsigned memoryRequests;
};

static IterationStats iterationStats(Pattern pattern, unsigned threads, unsigned stride) {
  IterationStats stats{stride, 0, 0, 0, 0};
  for (unsigned warp = 0; warp * WARP_SIZE < threads; ++warp) {
    const WarpAccesses accesses = warpAccesses(pattern, threads, stride, warp);
    const bool warpIsActive = accesses.activeLanes > 0;
    const bool warpIsDivergent = warpIsActive && accesses.activeLanes < WARP_SIZE;
    const unsigned readOwnRequests = static_cast<unsigned>(accesses.ownSegments.size());
    const unsigned readPartnerRequests = static_cast<unsigned>(accesses.partnerSegments.size());
    const unsigned writeOwnRequests = static_cast<unsigned>(accesses.ownSegments.size());

    stats.activeThreads += accesses.activeLanes;
    stats.activeWarps += warpIsActive ? 1 : 0;
    stats.divergentWarps += warpIsDivergent ? 1 : 0;
    stats.memoryRequests += readOwnRequests + readPartnerRequests + writeOwnRequests;
  }
  return stats;
}

struct PatternTotals {
  unsigned resources;
  unsigned committedResults;
  unsigned memoryRequests;
};

static PatternTotals patternTotals(Pattern pattern, unsigned threads) {
  PatternTotals totals{0, 0, 0};
  for (unsigned stride : stridesOf(pattern, threads)) {
    const IterationStats stats = iterationStats(pattern, threads, stride);
    totals.resources += stats.activeWarps * WARP_SIZE;
    totals.committedResults += stats.activeThreads;
    totals.memoryRequests += stats.memoryRequests;
  }
  return totals;
}

static unsigned sharedMemoryKernelRequests(unsigned threads) {
  unsigned loadRequests = 0;
  for (unsigned warp = 0; warp * WARP_SIZE < threads; ++warp) {
    std::set<unsigned> firstHalfSegments;
    std::set<unsigned> secondHalfSegments;
    for (unsigned lane = 0; lane < WARP_SIZE && warp * WARP_SIZE + lane < threads; ++lane) {
      const unsigned thread = warp * WARP_SIZE + lane;
      firstHalfSegments.insert(memorySegmentOf(thread));
      secondHalfSegments.insert(memorySegmentOf(thread + threads));
    }
    loadRequests += static_cast<unsigned>(firstHalfSegments.size() + secondHalfSegments.size());
  }
  const unsigned finalStoreRequests = 1;
  return loadRequests + finalStoreRequests;
}

static double efficiency(const PatternTotals& totals) {
  return static_cast<double>(totals.committedResults) / totals.resources;
}

static void printResourceTable(unsigned threads) {
  std::printf("\nExecution resources, %u elements = %u threads = %u warps\n",
              2 * threads,
              threads,
              threads / WARP_SIZE);
  std::printf("%5s | %-30s | %-30s\n",
              "iter",
              "simple: stride threads warps",
              "convergent: stride threads warps");
  const std::vector<unsigned> simpleStrides = stridesOf(Pattern::Simple, threads);
  const std::vector<unsigned> convergentStrides = stridesOf(Pattern::Convergent, threads);
  for (size_t k = 0; k < simpleStrides.size(); ++k) {
    const IterationStats simple = iterationStats(Pattern::Simple, threads, simpleStrides[k]);
    const IterationStats convergent =
        iterationStats(Pattern::Convergent, threads, convergentStrides[k]);
    std::printf("%5zu | %8u %8u %6u%7s | %8u %8u %6u\n",
                k + 1,
                simple.stride,
                simple.activeThreads,
                simple.activeWarps,
                "",
                convergent.stride,
                convergent.activeThreads,
                convergent.activeWarps);
  }
}

static bool checkExecutionResources() {
  printResourceTable(BOOK_ANALYSIS_THREADS);
  const PatternTotals simple = patternTotals(Pattern::Simple, BOOK_ANALYSIS_THREADS);
  const PatternTotals convergent = patternTotals(Pattern::Convergent, BOOK_ANALYSIS_THREADS);
  const PatternTotals reversed = patternTotals(Pattern::Reversed, BOOK_ANALYSIS_THREADS);

  std::printf("resources consumed (active warps x 32): simple %u, convergent %u, reversed %u\n",
              simple.resources,
              convergent.resources,
              reversed.resources);
  std::printf("results committed (active threads):     simple %u, convergent %u, reversed %u\n",
              simple.committedResults,
              convergent.committedResults,
              reversed.committedResults);
  std::printf("efficiency: simple %u/%u = %.2f, convergent %u/%u = %.2f (%.0f%%)\n",
              simple.committedResults,
              simple.resources,
              efficiency(simple),
              convergent.committedResults,
              convergent.resources,
              efficiency(convergent),
              100.0 * efficiency(convergent));

  const bool ok = simple.resources == BOOK_SIMPLE_RESOURCES &&
                  convergent.resources == BOOK_CONVERGENT_RESOURCES &&
                  simple.committedResults == BOOK_COMMITTED_RESULTS &&
                  convergent.committedResults == BOOK_COMMITTED_RESULTS &&
                  reversed.resources == BOOK_CONVERGENT_RESOURCES;
  std::printf("book: 864 and 384 resources, 255 results: %s\n", passOrFail(ok));
  return ok;
}

static bool checkMemoryRequests() {
  const PatternTotals simple = patternTotals(Pattern::Simple, BOOK_ANALYSIS_THREADS);
  const PatternTotals convergent = patternTotals(Pattern::Convergent, BOOK_ANALYSIS_THREADS);
  const PatternTotals reversed = patternTotals(Pattern::Reversed, BOOK_ANALYSIS_THREADS);
  const unsigned sharedMemory = sharedMemoryKernelRequests(BOOK_ANALYSIS_THREADS);

  std::printf("\nGlobal memory requests, %u elements (128-byte segments per warp access)\n",
              BOOK_ANALYSIS_ELEMENTS);
  std::printf("  simple %u, convergent %u, reversed (ex. 3) %u, shared memory %u (= 256/32 + 1)\n",
              simple.memoryRequests,
              convergent.memoryRequests,
              reversed.memoryRequests,
              sharedMemory);

  const bool ok = simple.memoryRequests == BOOK_SIMPLE_REQUESTS &&
                  convergent.memoryRequests == BOOK_CONVERGENT_REQUESTS &&
                  sharedMemory == BOOK_SHARED_MEMORY_REQUESTS;
  std::printf("book: 141, 36 and 9: %s\n", passOrFail(ok));
  return ok;
}

static bool checkDivergenceExercises() {
  const unsigned threads = DIVERGENCE_EXERCISE_THREADS;
  std::printf(
      "\nExercises 1 and 2: %u elements = %u threads = %u warps; divergent warp = some but"
      " not all lanes active\n",
      DIVERGENCE_EXERCISE_ELEMENTS,
      threads,
      threads / WARP_SIZE);
  std::printf("%5s | %-34s | %-34s\n",
              "iter",
              "simple: stride threads warps diverg",
              "convergent: stride threads warps diverg");

  const std::vector<unsigned> simpleStrides = stridesOf(Pattern::Simple, threads);
  const std::vector<unsigned> convergentStrides = stridesOf(Pattern::Convergent, threads);
  unsigned exerciseOneAnswer = 0;
  unsigned exerciseTwoAnswer = 0;
  for (size_t k = 0; k < simpleStrides.size(); ++k) {
    const IterationStats simple = iterationStats(Pattern::Simple, threads, simpleStrides[k]);
    const IterationStats convergent =
        iterationStats(Pattern::Convergent, threads, convergentStrides[k]);
    const bool isFifthIteration = k == FIFTH_ITERATION_INDEX;
    if (isFifthIteration) {
      exerciseOneAnswer = simple.divergentWarps;
      exerciseTwoAnswer = convergent.divergentWarps;
    }
    std::printf("%5zu | %8u %8u %6u %6u%5s | %8u %8u %6u %6u%s\n",
                k + 1,
                simple.stride,
                simple.activeThreads,
                simple.activeWarps,
                simple.divergentWarps,
                "",
                convergent.stride,
                convergent.activeThreads,
                convergent.activeWarps,
                convergent.divergentWarps,
                isFifthIteration ? "   <- 5th iteration" : "");
  }

  std::printf("exercise 1 (simple, 5th iteration, stride 16): %u divergent warps\n",
              exerciseOneAnswer);
  std::printf("exercise 2 (convergent, 5th iteration, stride 32): %u divergent warps\n",
              exerciseTwoAnswer);
  return exerciseOneAnswer == EXPECTED_SIMPLE_DIVERGENT_WARPS &&
         exerciseTwoAnswer == EXPECTED_CONVERGENT_DIVERGENT_WARPS;
}

static bool checkChapterAnalysis() {
  const bool resourcesOk = checkExecutionResources();
  const bool requestsOk = checkMemoryRequests();
  const bool exercisesOk = checkDivergenceExercises();
  return resourcesOk && requestsOk && exercisesOk;
}

struct TraceRun {
  std::vector<float> rows;
  float output;
};

template <class Kernel>
static TraceRun runTracedKernel(Kernel kernel) {
  float* input = allocateOnDevice<float>(TRACE_ELEMENTS);
  float* output = allocateOnDevice<float>(1);
  float* trace = allocateOnDevice<float>(TRACE_ITERATIONS * TRACE_ELEMENTS);
  copyToDevice(input, TRACE_INPUT, TRACE_ELEMENTS);

  launch(kernel, dim3(1), dim3(TRACE_THREADS), input, output, trace);
  CHECK(cudaGetLastError());

  TraceRun run;
  run.rows.resize(TRACE_ITERATIONS * TRACE_ELEMENTS);
  copyToHost(run.rows.data(), trace, run.rows.size());
  run.output = readFromDevice(output);

  CHECK(cudaFree(input));
  CHECK(cudaFree(output));
  CHECK(cudaFree(trace));
  return run;
}

static void replayIterationOnHost(Pattern pattern, unsigned stride, std::vector<float>& values) {
  for (unsigned thread = 0; thread < TRACE_THREADS; ++thread) {
    if (isActive(pattern, thread, stride, TRACE_THREADS)) {
      const unsigned owned = ownedPosition(pattern, thread, TRACE_THREADS);
      const unsigned partner = partnerPosition(pattern, thread, stride, TRACE_THREADS);
      values[owned] += values[partner];
    }
  }
}

static void printTraceRow(const char* label, const float* values) {
  std::printf("    %-16s", label);
  for (unsigned k = 0; k < TRACE_ELEMENTS; ++k) {
    std::printf(" %3.0f", values[k]);
  }
  std::printf("\n");
}

template <class Kernel>
static bool checkTrace(const char* title, Kernel kernel, Pattern pattern) {
  const TraceRun run = runTracedKernel(kernel);
  std::vector<float> expected(TRACE_INPUT, TRACE_INPUT + TRACE_ELEMENTS);

  std::printf("  %s\n", title);
  printTraceRow("input:", TRACE_INPUT);
  bool ok = true;
  const std::vector<unsigned> strides = stridesOf(pattern, TRACE_THREADS);
  for (unsigned iteration = 0; iteration < TRACE_ITERATIONS; ++iteration) {
    replayIterationOnHost(pattern, strides[iteration], expected);
    const float* traced = &run.rows[iteration * TRACE_ELEMENTS];
    char label[32];
    std::snprintf(label, sizeof label, "after stride %u:", strides[iteration]);
    printTraceRow(label, traced);
    for (unsigned k = 0; k < TRACE_ELEMENTS; ++k) {
      ok = ok && traced[k] == expected[k];
    }
  }

  ok = ok && run.output == TRACE_TOTAL;
  std::printf("    output %.0f  %s\n", run.output, passOrFail(ok));
  return ok;
}

static bool checkExerciseSixTraces() {
  std::printf(
      "\nExercise 6: {6, 2, 7, 4, 5, 8, 3, 1}, one block of 4 threads, array after each "
      "iteration\n");
  const bool simpleOk = checkTrace(
      "a. SimpleSumReductionKernel (traced copy)", SimpleSumReductionKernelTraced, Pattern::Simple);
  const bool convergentOk = checkTrace("b. ConvergentSumReductionKernel (traced copy)",
                                       ConvergentSumReductionKernelTraced,
                                       Pattern::Convergent);
  const bool reversedOk = checkTrace("ex. 3 ReversedConvergentSumReductionKernel (traced copy)",
                                     ReversedConvergentSumReductionKernelTraced,
                                     Pattern::Reversed);
  return simpleOk && convergentOk && reversedOk;
}

int main() {
  printBanner();

  const InputData data = makeInputData();
  const DeviceBuffers device = allocateDeviceBuffers(data);

  printTableHeader();
  const bool singleBlockOk = checkSingleBlockKernels(data, device);
  const bool multiBlockOk = checkMultiBlockKernels(data, device);
  const bool analysisOk = checkChapterAnalysis();
  const bool tracesOk = checkExerciseSixTraces();
  freeDeviceBuffers(device);

  const bool ok = singleBlockOk && multiBlockOk && analysisOk && tracesOk;
  std::printf("\n%s\n", ok ? "ALL PASS" : "SOME FAILED");
  if (IS_CPU_EMULATION) {
    std::printf("(CPU emulation: the times measure thread start-up and barriers, not the GPU.)\n");
  }
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. reduction_book.cu -o reduction_book && ./reduction_book